In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Created on 2024-05-27

@author: Juan Enrique López

@description: Jupyter Notebook creado para descargar las reglas de diferentes fuentes y clasificarlas por TTP

"""

In [1]:
from attackcti import attack_client
import pandas as pd
import re
import wget
import zipfile

import os
from os import remove


from shutil import rmtree
import shutil
from toml import TomlDecodeError
import re
import csv
import yaml
import toml
import yara

from collections import Counter

import numpy as np
import sys
import json

In [2]:
#temp_path = os.environ['TEMP']

temp_path = r'C:\Users\jelopez\Downloads'

path_actual = os.getcwd()

save_path = os.path.join(path_actual, 'saved')

In [3]:
# En primer lugar creamos la carpeta donde vamos a guardar las TTP's asociadas a esta fuente
def create_output_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [4]:
def download_unzip(url_zip, temp_path):
    #Descarga el fichero pasado por parametro
    wget.download(url_zip, os.path.join(temp_path, 'temp.zip'))
    #Descomprimir el fichero descargado y se borra el fichero descargado
    f_zip = zipfile.ZipFile(os.path.join(temp_path, 'temp.zip'))
    try:
        f_zip.extractall(path=temp_path)
    except:
        print('ERROR: MAX_PATH longer than 256 characters')
    f_zip.close()
    remove(os.path.join(temp_path, 'temp.zip'))

In [5]:
def download_folder_unzip(ruta, folder, temp_path):
    try:
        archivo_zip = wget.download(ruta, os.path.join(temp_path, 'temp.zip'))
        with zipfile.ZipFile(archivo_zip, 'r') as f_zip:
            for archivo in f_zip.namelist():
                if archivo.startswith(folder):
                    f_zip.extract(archivo, path=temp_path)
        os.remove(archivo_zip)
        print("Descarga y descompresión completadas correctamente.")
    except Exception as e:
        print('Error:', e)

In [6]:
def getListOfFilesSub(dirName):
    # Crea lista de ficheros y subdirectorios
    listOfFile = os.listdir(dirName)
    allFiles = list()
    #Recorre los directorios y subdirectorios y genera una lista con los ficheros encontrados en estos
    for entry in listOfFile:
        fullPath = os.path.join(dirName, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + getListOfFilesSub(fullPath)
        else:
            allFiles.append(fullPath)
    #Devuelve la lista con los ficheros
    return allFiles

In [7]:
def get_folders(path):
    if not os.path.exists(path):
        raise ValueError("La ruta proporcionada no existe.")
    if not os.path.isdir(path):
        raise ValueError("La ruta proporcionada no es un directorio.")
    folders = [name for name in os.listdir(path) if os.path.isdir(os.path.join(path, name))]
    return folders

In [8]:
def techniques():
    lift = attack_client()
    #Solicitud a la libreria attackcti la extraccion de las tecnicas Enterprise y normalizar el json en un dataframe
    techniques = lift.get_enterprise_techniques(stix_format=False)
    techniques = pd.json_normalize(techniques)
    #Eliminar las tecnicas deprecadas y revocadas
    techniques = techniques[(techniques['mitre_deprecated'] != True)]
    # Eliminamos duplicados y convertimos en lista
    techniques = techniques['technique_id'].drop_duplicates().tolist()
    techniques = sorted(techniques, key=len, reverse=True)
    return techniques

In [9]:
techniques_enterprise = techniques()
techniques_enterprise[0:3]

[taxii2client.v20] [WARNING ] [2024-05-28 12:31:44,653] TAXII Server Response did not include 'Content-Range' header - results could be incomplete.
[taxii2client.v20] [WARNING ] [2024-05-28 12:31:44,677] TAXII Server Response with different amount of objects! Setting per_request=780


['T1059.010', 'T1564.012', 'T1027.013']

In [10]:
# Buscamos en la columna de descripción o informativa
def find_techniques(texto):
    if isinstance(texto, str):
        found = []
        for item in techniques_enterprise:
            if item in texto:
                found.append(item)
        return ', '.join(found)
    else:
        return ''

In [11]:
def find_techniques_in_list(cadena, techniques_enterprise):
    for item in techniques_enterprise:
        if item in cadena:
            return item
    return None

In [12]:
def clean_names(name):
    delete_chars = r'[\/\\|,";:#\[\]]'
    name = re.sub(delete_chars, '', name)
    name = " ".join(name.split())
    return name.strip()

In [13]:
def copy_file_to_path(item_path, save_path, ttp_name):
    if not os.path.exists(os.path.join(save_path, ttp_name)):
        os.makedirs(os.path.join(save_path, ttp_name))
    shutil.copy(item_path,os.path.join(save_path, ttp_name))
    print('Copiado correctamente a la carpeta de la TTP: '+ ttp_name) 

In [14]:
class NotTTPatTags(Exception):
    def __init__(self, mensaje):
        self.mensaje = mensaje
        super().__init__(self.mensaje)

In [15]:
def get_unique_items(list_paths):
    from pathlib import Path
    unique_list = []
    for item_path in list_paths:
        unique_list.append(Path(item_path).name)
    unique_list = list(set(unique_list))
    return unique_list


In [16]:
def copy_files_between_paths(path_source, path_destiny):
    if not os.path.exists(path_source):
        raise ValueError(f"El directorio de origen '{path_source}' no existe.")
    if not os.path.isdir(path_source):
        raise ValueError(f"'{path_source}' no es un directorio.")
    # Si el directorio de destino no existe, crearlo
    if not os.path.exists(path_destiny):
        os.makedirs(path_destiny)
    for item in os.listdir(path_source):
        if item != '.DS_Store': 
            src_path = os.path.join(path_source, item)
            dst_path = os.path.join(path_destiny, item)
            if os.path.isdir(src_path):
                shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            else:
                shutil.copy2(src_path, dst_path)

In [17]:
#Función para devolver el numero de reglas que han sido asignadas a 2 o más ttps

def count_duplicate_rules(paths):
    # Extraer nombres de archivo de las rutas
    filenames = [os.path.basename(path) for path in paths]
    # Contar archivos repetidos
    file_counts = Counter(filenames)
    # Obtener solo los archivos que están repetidos
    duplicates = {filename: count for filename, count in file_counts.items() if count > 1}
    return duplicates

In [16]:
# path_techniques = 'C:/Users/jelopez/Documents/CyberProof/python/recursos/id_tecnicas_16052024.csv'
# techniques_enterprise = pd.read_csv(path_techniques)
# techniques_enterprise = techniques_enterprise['ID Tecnica'].to_list()
# techniques_enterprise = sorted(techniques_enterprise, key=len, reverse=True)
# techniques_enterprise[:3]

['T1059.010', 'T1564.012', 'T1027.013']

#### TTP's Mappings

In [18]:
zip_mappings = 'https://github.com/center-for-threat-informed-defense/security-stack-mappings/archive/refs/heads/main.zip'
save_path_mappings = os.path.join(save_path, 'mappings')
unzip_folder = r'security-stack-mappings-main/mappings/'

In [20]:
download_folder_unzip(zip_mappings, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [22]:
create_output_folder(save_path_mappings)

In [23]:
files = getListOfFilesSub(os.path.join(temp_path, unzip_folder))
len(files)

343

In [ ]:
# Vamos a utilizar el mismo metodo que en yaras y atomic de tal forma que leemos todo el contenido del archivo y mapeamos con las técnicas disponibles

In [25]:
num_yamls = []
ttp_in_filename = []
content_in_yaml = []
no_content_in_yaml = []
ttp_in_yaml = []
error_load = []
no_ttp_in_content_or_filename = []


for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yaml' or extension == '.yml':
        with open(item, 'r', encoding='utf-8') as file:
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                yaml_content = file.read()
                yaml_content = yaml_content.upper()
                num_yamls.append(item)
                # Revisamos que se haya podido obtener texto del contenido del .yaml 
                if isinstance(yaml_content, str):
                    ttps = find_techniques(yaml_content)
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    content_in_yaml.append(item)
                else:
                    #Si el contenido no dispone de texto, buscamos ttps en el propio nombre del fichero
                    ttps = find_techniques(item.upper())
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    no_content_in_yaml.append(item)
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_yaml.append(item)
                            copy_file_to_path(item, save_path_mappings, ttp)
                
                #En cualquier caso, siempre buscamos ttp en el nombre del fichero
                ttps = find_techniques(item.upper())
                ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_filename.append(item)
                            copy_file_to_path(item, save_path_mappings, ttp)
                else:
                    copy_file_to_path(item, save_path_mappings, 'T0000')

            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except Exception as e:
                print(f"Error al intentar leer el archivo markdown: {e}")
                error_load.append(item)
                ttps = find_techniques(item.upper())
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_filename.append(item)
                            copy_file_to_path(item, save_path_mappings, ttp)
                else:
                    copy_file_to_path(item, save_path_mappings, 'T0000')

Copiado correctamente a la carpeta de la TTP: T1078.004
Copiado correctamente a la carpeta de la TTP: T1110.004
Copiado correctamente a la carpeta de la TTP: T1110.003
Copiado correctamente a la carpeta de la TTP: T1110.002
Copiado correctamente a la carpeta de la TTP: T1110.001
Copiado correctamente a la carpeta de la TTP: T1110
Copiado correctamente a la carpeta de la TTP: T1078
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T1562.008
Copiado correctamente a la carpeta de la TTP: T1595.002
Copiado correctamente a la carpeta de la TTP: T1595.001
Copiado correctamente a la carpeta de la TTP: T1098.004
Copiado correctamente a la carpeta de la TTP: T1562.006
Copiado correctamente a la carpeta de la TTP: T1071.004
Copiado correctamente a la carpeta de la TTP: T1071.003
Copiado correctamente a la carpeta de la TTP: T1071.002
Copiado correctamente a la carpeta de la TTP: T1071.001
Copiado 

In [29]:
print('Reglas totales para asignar: '+str(len(num_yamls)))
rules_unique = get_unique_items(num_yamls)
print('Reglas únicas para asignar: '+str(len(rules_unique)))
print('Reglas que retornan error de lectura: '+str(len(error_load)))

Reglas totales para asignar: 126
Reglas únicas para asignar: 126
Reglas que retornan error de lectura: 0


In [27]:
print('El archivo ha sido abierto y dispone de contenido: '+str(len(content_in_yaml)))
print('El archivo ha sido abierto y NO dispone de contenido: '+str(len(no_content_in_yaml)))
print('Se ha podido identificar al menos una TTP en el contenido del archivo: '+str(len(ttp_in_yaml)))
print('Se ha podido identificar al menos una TTP en el nombre del archivo: '+str(len(ttp_in_filename)))

El archivo ha sido abierto y dispone de contenido: 126
El archivo ha sido abierto y NO dispone de contenido: 0
Se ha podido identificar al menos una TTP en el contenido del archivo: 1846
Se ha podido identificar al menos una TTP en el nombre del archivo: 0


In [28]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_mappings))))
try:
    t0000_count = len(getListOfFilesSub(os.path.join(save_path_mappings, 'T0000')))
    print('Reglas sin TTP asignada (T0000):', t0000_count)
except FileNotFoundError:
    t0000_count = 0
    print('Reglas sin TTP asignada (T0000): 0')
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_mappings)) - t0000_count))
print('Reglas asignadas a 2 o más TTP: '+str(len(count_duplicate_rules(getListOfFilesSub(save_path_mappings)))))

Número de TTPs únicas identificadas (puede incluir T0000): 407
Reglas sin TTP asignada (T0000): 126
Reglas asignadas a TTP: 1846
Reglas asignadas a 2 o más TTP: 108


#### TTP's Azure/Azure-Sentinel/tree/master/Hunting Queries

In [ ]:
path_hunting = 'https://github.com/Azure/Azure-Sentinel/tree/master/Hunting%20Queries'

save_path_hunting = os.path.join(save_path, 'Azure-Sentinel Hunting Queries')

zip_sentinel = 'https://github.com/Azure/Azure-Sentinel/archive/refs/heads/master.zip'

In [ ]:
create_output_folder(save_path_hunting)

In [ ]:
download_unzip(zip_sentinel, temp_path)

In [30]:
files = getListOfFilesSub(temp_path+r'\Azure-Sentinel-master\Hunting Queries')

['C:\\Users\\jelopez\\AppData\\Local\\Temp\\Azure-Sentinel-master\\Hunting Queries\\ASimProcess\\Discorddownloadinvokedfromcmdline(ASIMVersion).yaml',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\Azure-Sentinel-master\\Hunting Queries\\ASimProcess\\imProcess_Certutil-LOLBins.yaml',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\Azure-Sentinel-master\\Hunting Queries\\ASimProcess\\imProcess_cscript_summary.yaml',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\Azure-Sentinel-master\\Hunting Queries\\ASimProcess\\imProcess_Dev-0056CommandLineActivityNovember2021(ASIMVersion).yaml',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\Azure-Sentinel-master\\Hunting Queries\\ASimProcess\\imProcess_enumeration_user_and_group.yaml',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\Azure-Sentinel-master\\Hunting Queries\\ASimProcess\\imProcess_ExchangePowerShellSnapin.yaml',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\Azure-Sentinel-master\\Hunting Queries\\ASimProcess\\imProcess_HostExportingMailboxAndRemovingExp

In [70]:
for item in files:
    root,extension = os.path.splitext(item)
    if extension == '.yaml':
        with open(item) as file:
            documents = yaml.full_load(file)
            ttp = documents.get('relevantTechniques', [])
            print(item)
            if isinstance(ttp, list) and len(ttp)>=1:
                for i in range(len(ttp)):
                    if not os.path.exists(os.path.join(save_path_hunting, ttp[i])):
                        os.makedirs(os.path.join(save_path_hunting, ttp[i]))
                    shutil.copy(item,os.path.join(save_path_hunting, ttp[i]))
                    print('Copiado correctamente')
                    
                

C:\Users\jelopez\AppData\Local\Temp\Azure-Sentinel-master\Hunting Queries\ASimProcess\Discorddownloadinvokedfromcmdline(ASIMVersion).yaml
Copiado correctamente
Copiado correctamente
Copiado correctamente
C:\Users\jelopez\AppData\Local\Temp\Azure-Sentinel-master\Hunting Queries\ASimProcess\imProcess_Certutil-LOLBins.yaml
Copiado correctamente
C:\Users\jelopez\AppData\Local\Temp\Azure-Sentinel-master\Hunting Queries\ASimProcess\imProcess_cscript_summary.yaml
C:\Users\jelopez\AppData\Local\Temp\Azure-Sentinel-master\Hunting Queries\ASimProcess\imProcess_Dev-0056CommandLineActivityNovember2021(ASIMVersion).yaml
Copiado correctamente
C:\Users\jelopez\AppData\Local\Temp\Azure-Sentinel-master\Hunting Queries\ASimProcess\imProcess_enumeration_user_and_group.yaml
C:\Users\jelopez\AppData\Local\Temp\Azure-Sentinel-master\Hunting Queries\ASimProcess\imProcess_ExchangePowerShellSnapin.yaml
Copiado correctamente
C:\Users\jelopez\AppData\Local\Temp\Azure-Sentinel-master\Hunting Queries\ASimProcess\i

##### TTP's BlueTeamLabs/sentinel-attack

In [87]:
zip_sentinel_attack = 'https://github.com/netevert/sentinel-attack/archive/refs/heads/master.zip'

save_path_sentinel_attack = os.path.join(save_path, 'netevert_sentinel-attack')

In [88]:
create_output_folder(save_path_sentinel_attack)

'c:\\Users\\jelopez\\Documents\\CyberProof\\python\\download_ttps\\saved\\netevert_sentinel-attack'

In [74]:
download_unzip(zip_sentinel_attack, temp_path)

In [75]:
files = getListOfFilesSub(temp_path+r'\sentinel-attack-master')

['C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\.gitignore',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\azuredeploy.json',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\CODE_OF_CONDUCT.md',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\CONTRIBUTING.md',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\deployment\\gallery.azuredeploy.json',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\detections\\sentinel_attack_rules.json',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\detections\\T0000_Console_History.txt',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\detections\\T0000_Named_Pipes.txt',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\detections\\T0000_Named_Pipes_CobaltStrike.txt',
 'C:\\Users\\jelopez\\AppData\\Local\\Temp\\sentinel-attack-master\\detections\\T0000_Remotely_Query_Login_Sessi

In [96]:
for item in files:
    root,extension = os.path.splitext(item)
    if extension == '.txt':
        basename = os.path.basename(item)
        print(basename[0:5])
        if basename[0] == 'T' and basename[1:5].isdigit():
            if not os.path.exists(os.path.join(save_path_sentinel_attack, basename[0:5])):
                os.makedirs(os.path.join(save_path_sentinel_attack, basename[0:5]))
                print('Carpeta creada correctamente')
            #print(os.path.join(save_path_sentinel_attack, basename[0:5]))
            shutil.copy(item,os.path.join(save_path_sentinel_attack, basename[0:5]))

T0000
Carpeta creada correctamente
T0000
T0000
T0000
T0000
T0000
T1002
Carpeta creada correctamente
T1003
Carpeta creada correctamente
T1003
T1003
T1003
T1003
T1004
Carpeta creada correctamente
T1007
Carpeta creada correctamente
T1012
Carpeta creada correctamente
T1012
T1013
Carpeta creada correctamente
T1015
Carpeta creada correctamente
T1015
T1016
Carpeta creada correctamente
T1018
Carpeta creada correctamente
T1018
T1027
Carpeta creada correctamente
T1028
Carpeta creada correctamente
T1031
Carpeta creada correctamente
T1033
Carpeta creada correctamente
T1036
Carpeta creada correctamente
T1036
T1037
Carpeta creada correctamente
T1040
Carpeta creada correctamente
T1042
Carpeta creada correctamente
T1044
Carpeta creada correctamente
T1047
Carpeta creada correctamente
T1047
T1047
T1047
T1047
T1049
Carpeta creada correctamente
T1050
Carpeta creada correctamente
T1053
Carpeta creada correctamente
T1053
T1054
Carpeta creada correctamente
T1054
T1055
Carpeta creada correctamente
T1057
Carpe

T1070
Carpeta creada correctamente
T1074
Carpeta creada correctamente
T1076
Carpeta creada correctamente
T1076
T1077
Carpeta creada correctamente
T1077
T1077
T1081
Carpeta creada correctamente
T1082
Carpeta creada correctamente
T1085
Carpeta creada correctamente
T1086
Carpeta creada correctamente
T1086
T1087
Carpeta creada correctamente
T1088
Carpeta creada correctamente
T1088
T1089
Carpeta creada correctamente
T1093
Carpeta creada correctamente
T1096
Carpeta creada correctamente
T1103
Carpeta creada correctamente
T1107
Carpeta creada correctamente
T1112
Carpeta creada correctamente
T1115
Carpeta creada correctamente
T1117
Carpeta creada correctamente
T1117
T1118
Carpeta creada correctamente
T1121
Carpeta creada correctamente
T1122
Carpeta creada correctamente
T1123
Carpeta creada correctamente
T1124
Carpeta creada correctamente
T1126
Carpeta creada correctamente
T1127
Carpeta creada correctamente
T1128
Carpeta creada correctamente
T1128
T1130
Carpeta creada correctamente
T1131
Carpeta

##### TTP's Elastic - 1

In [115]:

zip_elastic_1 = 'https://github.com/elastic/detection-rules/archive/refs/heads/main.zip'

save_path_elastic_1 = os.path.join(save_path, 'elastic_1')

In [ ]:
create_output_folder(save_path_elastic_1)

In [120]:
download_folder_unzip(zip_elastic_1, 'detection-rules-main/rules', temp_path)

Descarga y descompresión completadas correctamente.


In [121]:
files = getListOfFilesSub(temp_path+r'\detection-rules-main/rules')
files[:3]

['C:\\Users\\jelopez\\Downloads\\detection-rules-main/rules\\apm\\apm_403_response_to_a_post.toml',
 'C:\\Users\\jelopez\\Downloads\\detection-rules-main/rules\\apm\\apm_405_response_method_not_allowed.toml',
 'C:\\Users\\jelopez\\Downloads\\detection-rules-main/rules\\apm\\apm_sqlmap_user_agent.toml',
 'C:\\Users\\jelopez\\Downloads\\detection-rules-main/rules\\cross-platform\\command_and_control_google_drive_malicious_file_download.toml',
 'C:\\Users\\jelopez\\Downloads\\detection-rules-main/rules\\cross-platform\\command_and_control_non_standard_ssh_port.toml',
 'C:\\Users\\jelopez\\Downloads\\detection-rules-main/rules\\cross-platform\\credential_access_cookies_chromium_browsers_debugging.toml',
 'C:\\Users\\jelopez\\Downloads\\detection-rules-main/rules\\cross-platform\\defense_evasion_agent_spoofing_mismatched_id.toml',
 'C:\\Users\\jelopez\\Downloads\\detection-rules-main/rules\\cross-platform\\defense_evasion_agent_spoofing_multiple_hosts.toml',
 'C:\\Users\\jelopez\\Downloads\

In [152]:
for item in files:
    root,extension = os.path.splitext(item)
    if extension == '.toml':
        with open(item, 'r') as file:
            rules = []
            try:
                data_toml = toml.load(file)
            except (IndexError, TomlDecodeError) as e:
                if isinstance(e, IndexError):
                    print("Se produjo un IndexError:", e)
                    print('No se ha podido asignar ninguna carpeta al archivo:  '+str(item))
                elif "Unterminated string found. Reached end of file." in str(e):
                    print('No se ha podido asignar ninguna carpeta al archivo:  '+str(item))
            except Exception as e:
                print("Se produjo un error:", e)
                print('No se ha podido asignar ninguna carpeta al archivo:  '+str(item))
            if len(data_toml.get('rule',[]).get('threat',[]))>0:
                for i in range(len(data_toml['rule']['threat'])):
                    #print(i)
                    if data_toml.get('rule', []).get('threat', [])[i].get('technique'):
                        data_rules = data_toml['rule']['threat'][i]['technique']
                        for i in range(len(data_rules)):
                            rules.append(data_rules[i]['id'])
                            subtechniques = data_rules[i].get('subtechnique', [])
                            if len(subtechniques)>0:
                                for z in range(len(subtechniques)):
                                    rules.append(subtechniques[z]['id'])
        for rule in rules:
            if not os.path.exists(os.path.join(save_path_elastic_1, rule)):
                os.makedirs(os.path.join(save_path_elastic_1, rule))
            shutil.copy(item,os.path.join(save_path_elastic_1, rule))

            #Guardar en algun sitio como no indexadas

Se produjo un IndexError: string index out of range
No se ha podido asignar ninguna carpeta al archivo:  C:\Users\jelopez\Downloads\detection-rules-main/rules\integrations\aws\collection_cloudtrail_logging_created.toml
Se produjo un IndexError: string index out of range
No se ha podido asignar ninguna carpeta al archivo:  C:\Users\jelopez\Downloads\detection-rules-main/rules\integrations\aws\defense_evasion_cloudtrail_logging_deleted.toml
Se produjo un IndexError: string index out of range
No se ha podido asignar ninguna carpeta al archivo:  C:\Users\jelopez\Downloads\detection-rules-main/rules\integrations\aws\defense_evasion_elasticache_security_group_creation.toml
Se produjo un IndexError: string index out of range
No se ha podido asignar ninguna carpeta al archivo:  C:\Users\jelopez\Downloads\detection-rules-main/rules\integrations\aws\defense_evasion_s3_bucket_configuration_deletion.toml
Se produjo un IndexError: string index out of range
No se ha podido asignar ninguna carpeta al

#### TTP's UCM Catalog 2024 Sentinel *from excel

In [381]:
# Esto es temporal ya que parece ser que he excedido el número de peticiones
#lift = attack_client()

path_techniques = 'C:/Users/jelopez/Documents/CyberProof/python/recursos/id_tecnicas_16052024.csv'
techniques_enterprise = pd.read_csv(path_techniques)
techniques_enterprise = techniques_enterprise['ID Tecnica'].to_list()
techniques_enterprise

['T1059.010',
 'T1564.012',
 'T1027.013',
 'T1574.014',
 'T1584.008',
 'T1548.006',
 'T1588.007',
 'T1218.015',
 'T1543.005',
 'T1665',
 'T1216.002',
 'T1556.009',
 'T1027.012',
 'T1036.009',
 'T1555.006',
 'T1016.002',
 'T1566.004',
 'T1598.004',
 'T1578.005',
 'T1659',
 'T1564.011',
 'T1657',
 'T1656',
 'T1567.004',
 'T1098.006',
 'T1654',
 'T1548.005',
 'T1653',
 'T1021.008',
 'T1562.012',
 'T1556.008',
 'T1652',
 'T1027.011',
 'T1027.010',
 'T1562.011',
 'T1552.008',
 'T1651',
 'T1650',
 'T1036.008',
 'T1567.003',
 'T1583.008',
 'T1021.007',
 'T1205.002',
 'T1608.006',
 'T1027.009',
 'T1027.008',
 'T1556.007',
 'T1546.016',
 'T1027.007',
 'T1593.003',
 'T1649',
 'T1070.009',
 'T1070.008',
 'T1584.007',
 'T1583.007',
 'T1070.007',
 'T1556.006',
 'T1586.003',
 'T1585.003',
 'T1648',
 'T1647',
 'T1622',
 'T1621',
 'T1505.005',
 'T1557.003',
 'T1059.009',
 'T1595.003',
 'T1098.005',
 'T1574.013',
 'T1556.005',
 'T1055.015',
 'T1564.010',
 'T1564.009',
 'T1559.003',
 'T1562.010',
 'T154

In [285]:
path_UCMCatalog2024 = r'C:\Users\jelopez\Documents\CyberProof\python\check_0905\UCM Catalog 2024'

path_UCMCatalog2024_Sentinel = path_UCMCatalog2024 + '/UCM Catalog 2024 [Sentinel].xlsx'

save_path_sentinel = os.path.join(save_path, 'UCM Catalog 2024 Sentinel')

In [ ]:
create_output_folder(save_path_sentinel)

In [297]:
# Formateo del fichero
UCMCatalog2024_Sentinel = pd.read_excel(path_UCMCatalog2024_Sentinel)
UCMCatalog2024_Sentinel['techniques'] = UCMCatalog2024_Sentinel['techniques'].str.replace('[','')
UCMCatalog2024_Sentinel['techniques'] = UCMCatalog2024_Sentinel['techniques'].str.replace(']','')
UCMCatalog2024_Sentinel['techniques'] = UCMCatalog2024_Sentinel['techniques'].str.replace('"','')
UCMCatalog2024_Sentinel['techniques'] = UCMCatalog2024_Sentinel['techniques'].str.split(',')
UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel.explode('techniques').reset_index()
UCMCatalog2024_Sentinel['query'] = UCMCatalog2024_Sentinel['query'].str.replace('\n', '', regex=False)
UCMCatalog2024_Sentinel['QUERY'] = UCMCatalog2024_Sentinel['query'].str.upper()
UCMCatalog2024_Sentinel['QUERY'] = UCMCatalog2024_Sentinel['QUERY'].fillna('N/A')
UCMCatalog2024_Sentinel.head(2)

,index,displayName,tactics,techniques,query,QUERY
0,0,Azure Redis Cache - Config modification attempt,"[""Execution""]",T0863,AzureActivity| where OperationNameValue contai...,AZUREACTIVITY| WHERE OPERATIONNAMEVALUE CONTAI...
1,1,Azure Redis Cache - Access keys regeneration a...,"[""ResourceDevelopment""]",T1587,AzureActivity| where OperationName == 'Regener...,AZUREACTIVITY| WHERE OPERATIONNAME == 'REGENER...


In [298]:
# Limpiamos el df para poder unir
UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel.assign(type_source='item raw')[['displayName','tactics','techniques','query','type_source']]
UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw[UCMCatalog2024_Sentinel_raw['techniques']!='']
UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw.dropna(subset=['techniques'])
UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw.drop_duplicates()
UCMCatalog2024_Sentinel_raw.rename(columns={'techniques': 'Technique','displayName':'Display name','tactics':'Tactics','query':'Query','type_source':'Type source'}, inplace=True)
UCMCatalog2024_Sentinel_raw.head(2)

,Display name,Tactics,Technique,Query,Type source
0,Azure Redis Cache - Config modification attempt,"[""Execution""]",T0863,AzureActivity| where OperationNameValue contai...,item raw
1,Azure Redis Cache - Access keys regeneration a...,"[""ResourceDevelopment""]",T1587,AzureActivity| where OperationName == 'Regener...,item raw


In [299]:
# Limpiamos y formateamos para la union
UCMCatalog2024_Sentinel_query = UCMCatalog2024_Sentinel
UCMCatalog2024_Sentinel_query['techniques_from_query'] = UCMCatalog2024_Sentinel_query['query'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Sentinel_query = UCMCatalog2024_Sentinel_query[UCMCatalog2024_Sentinel_query['techniques_from_query']!='']
UCMCatalog2024_Sentinel_query = UCMCatalog2024_Sentinel_query.dropna(subset=['techniques_from_query'])
UCMCatalog2024_Sentinel_query['techniques_from_query_explode'] = UCMCatalog2024_Sentinel_query['techniques_from_query'].str.split(',')
UCMCatalog2024_Sentinel_query = UCMCatalog2024_Sentinel_query.explode('techniques_from_query_explode')
UCMCatalog2024_Sentinel_query = UCMCatalog2024_Sentinel_query.drop_duplicates()
UCMCatalog2024_Sentinel_query = UCMCatalog2024_Sentinel_query.assign(type_source='item description')[['displayName','tactics','techniques_from_query_explode','query','type_source']].reset_index(drop=True)
UCMCatalog2024_Sentinel_query.rename(columns={'techniques_from_query_explode': 'Technique','displayName':'Display name','tactics':'Tactics','query':'Query','type_source':'Type source'}, inplace=True)
UCMCatalog2024_Sentinel_query.head(2)

,Display name,Tactics,Technique,Query,Type source
0,SOC - FW - Threat Intelligence - Malicious IP,"[""InitialAccess""]",T1597.001,// Threat Intelligence - Malicious IP// Mitre ...,item description
1,SOC - FW - Threat Intelligence - Malicious IP,"[""InitialAccess""]",T1597,// Threat Intelligence - Malicious IP// Mitre ...,item description


In [300]:
UCMCatalog2024_Sentinel_union = pd.concat([UCMCatalog2024_Sentinel_raw, UCMCatalog2024_Sentinel_query])
UCMCatalog2024_Sentinel_union.head(2)

,Display name,Tactics,Technique,Query,Type source
0,Azure Redis Cache - Config modification attempt,"[""Execution""]",T0863,AzureActivity| where OperationNameValue contai...,item raw
1,Azure Redis Cache - Access keys regeneration a...,"[""ResourceDevelopment""]",T1587,AzureActivity| where OperationName == 'Regener...,item raw


In [302]:

for indice, row in UCMCatalog2024_Sentinel_union.iterrows():
    folder_name = row['Technique'].replace(' ', '')
    file_name = row['Display name'] + ".csv"
    file_name = clean_names(file_name)
    folder_path = os.path.join(save_path_sentinel, folder_name)
    csv_file = os.path.join(save_path_sentinel, folder_name, file_name)
    os.makedirs(folder_path, exist_ok=True)
    #row.to_csv(csv_file, index=False, header=True, sep=';', quoting=csv.QUOTE_ALL)
    temp_df = pd.DataFrame(row).T
    temp_df.columns = UCMCatalog2024_Sentinel_union.columns
    temp_df.to_csv(csv_file, index=False, header=True, sep=';', quoting=csv.QUOTE_ALL)

#### TTP's UCM Catalog 2024 Qradar *from excel

In [303]:
# Esto es temporal ya que parece ser que he excedido el número de peticiones
#lift = attack_client()

path_techniques = 'C:/Users/jelopez/Documents/CyberProof/python/recursos/id_tecnicas_16052024.csv'
techniques_enterprise = pd.read_csv(path_techniques)
techniques_enterprise = techniques_enterprise['ID Tecnica'].to_list()

In [304]:
path_UCMCatalog2024 = r'C:\Users\jelopez\Documents\CyberProof\python\check_0905\UCM Catalog 2024'

path_UCMCatalog2024_Qradar = path_UCMCatalog2024 + '/UCM Catalog 2024 [Qradar].xlsx'

save_path_qradar = os.path.join(save_path, 'UCM Catalog 2024 Qradar')

In [ ]:
create_output_folder(save_path_qradar)

*baseline*

In [305]:
UCMCatalog2024_Qradar_baseline = pd.read_excel(path_UCMCatalog2024_Qradar,'Baseline')

UCMCatalog2024_Qradar_baseline['Mitre Technique'] = UCMCatalog2024_Qradar_baseline['Mitre Technique'].str.replace('\n', '', regex=False)

UCMCatalog2024_Qradar_baseline['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_baseline['Mitre Technique'].str.upper()
UCMCatalog2024_Qradar_baseline['techniques_from_raw'] = UCMCatalog2024_Qradar_baseline['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_baseline = UCMCatalog2024_Qradar_baseline.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_baseline['Required Telemetry'] = UCMCatalog2024_Qradar_baseline['Required Telemetry'].fillna('N/A')
UCMCatalog2024_Qradar_baseline['Name Mitre Technique'] = UCMCatalog2024_Qradar_baseline['Mitre Technique'].str.split('-').str[1]
UCMCatalog2024_Qradar_baseline['techniques_from_raw_explode'] = UCMCatalog2024_Qradar_baseline['techniques_from_raw'].str.split(',')
UCMCatalog2024_Qradar_baseline = UCMCatalog2024_Qradar_baseline.explode('techniques_from_raw_explode')
UCMCatalog2024_Qradar_baseline = UCMCatalog2024_Qradar_baseline[UCMCatalog2024_Qradar_baseline['techniques_from_raw']!='']
UCMCatalog2024_Qradar_baseline = UCMCatalog2024_Qradar_baseline.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_baseline.rename(columns={'techniques_from_raw_explode': 'Technique'}, inplace=True)

UCMCatalog2024_Qradar_baseline.shape

(124, 10)

In [306]:
UCMCatalog2024_Qradar_baseline.tail(2)

,Rule,Priority,Mitre Tactic,Mitre Technique,Log Source Type,Required Telemetry,MITRE_TECHNIQUE,techniques_from_raw,Name Mitre Technique,Technique
92,[CyberProof] - [Windows] - Suspicious MS Offic...,MED,Initial Access\nExecution\nDefense Evasion,T1059.003 - Command and Scripting Interpreter:...,Windows,N/A,T1059.003 - COMMAND AND SCRIPTING INTERPRETER:...,"T1059.003, T1566.001, T1566, T1218, T1059",Command and Scripting Interpreter: Windows Co...,T1059
100,[CyberProof] - [QRadar] - Multiple Failed Logins,LOW,Credential Access,T1110 - Brute Force,QRadar Audit,N/A,T1110 - BRUTE FORCE,T1110,Brute Force,T1110


In [307]:
colums_sel = ['Rule','Priority','Mitre Technique','Technique','Name Mitre Technique','Mitre Tactic','Required Telemetry']
UCMCatalog2024_Qradar_baseline = UCMCatalog2024_Qradar_baseline[colums_sel]
UCMCatalog2024_Qradar_baseline.tail(2)

,Rule,Priority,Mitre Technique,Technique,Name Mitre Technique,Mitre Tactic,Required Telemetry
92,[CyberProof] - [Windows] - Suspicious MS Offic...,MED,T1059.003 - Command and Scripting Interpreter:...,T1059,Command and Scripting Interpreter: Windows Co...,Initial Access\nExecution\nDefense Evasion,N/A
100,[CyberProof] - [QRadar] - Multiple Failed Logins,LOW,T1110 - Brute Force,T1110,Brute Force,Credential Access,N/A


In [308]:
for indice, row in UCMCatalog2024_Qradar_baseline.iterrows():
    folder_name = row['Technique'].replace(' ', '')
    file_name = 'Baseline - '+row['Rule'] + ".csv"
    file_name = clean_names(file_name)
    folder_path = os.path.join(save_path_qradar, folder_name)
    csv_file = os.path.join(save_path_qradar, folder_name, file_name)
    os.makedirs(folder_path, exist_ok=True)
    temp_df = pd.DataFrame(row).T
    temp_df.columns = UCMCatalog2024_Qradar_baseline.columns
    temp_df.to_csv(csv_file, index=False, header=True, sep=';', quoting=csv.QUOTE_ALL)

*Windows|Sysmon*

In [309]:
UCMCatalog2024_Qradar_winsysmon = pd.read_excel(path_UCMCatalog2024_Qradar,'Windows|Sysmon')
UCMCatalog2024_Qradar_winsysmon.shape

(223, 5)

In [310]:
UCMCatalog2024_Qradar_winsysmon = pd.read_excel(path_UCMCatalog2024_Qradar,'Windows|Sysmon')

UCMCatalog2024_Qradar_winsysmon['Mitre Technique'] = UCMCatalog2024_Qradar_winsysmon['Mitre Technique'].str.replace('\n', '', regex=False)

UCMCatalog2024_Qradar_winsysmon['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_winsysmon['Mitre Technique'].str.upper()
UCMCatalog2024_Qradar_winsysmon['techniques_from_raw'] = UCMCatalog2024_Qradar_winsysmon['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_winsysmon = UCMCatalog2024_Qradar_winsysmon.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_winsysmon['Mitre Tactic'] = UCMCatalog2024_Qradar_winsysmon['Mitre Tactic'].fillna('N/A')
UCMCatalog2024_Qradar_winsysmon['Name Mitre Technique'] = UCMCatalog2024_Qradar_winsysmon['Mitre Technique'].str.split('-').str[1]
UCMCatalog2024_Qradar_winsysmon['techniques_from_raw_explode'] = UCMCatalog2024_Qradar_winsysmon['techniques_from_raw'].str.split(',')
UCMCatalog2024_Qradar_winsysmon = UCMCatalog2024_Qradar_winsysmon.explode('techniques_from_raw_explode')
UCMCatalog2024_Qradar_winsysmon = UCMCatalog2024_Qradar_winsysmon[UCMCatalog2024_Qradar_winsysmon['techniques_from_raw']!='']
UCMCatalog2024_Qradar_winsysmon = UCMCatalog2024_Qradar_winsysmon.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_winsysmon.rename(columns={'techniques_from_raw_explode': 'Technique'}, inplace=True)
UCMCatalog2024_Qradar_winsysmon['Name Mitre Technique'] = UCMCatalog2024_Qradar_winsysmon['Name Mitre Technique'].fillna('N/A')
UCMCatalog2024_Qradar_winsysmon.head(2)

,Rule,Priority,Mitre Tactic,Mitre Technique,Log Source Type,MITRE_TECHNIQUE,techniques_from_raw,Name Mitre Technique,Technique
81,PingCastle Binary Execution,HIGH,"Execution, Defense Evasion","T1059, T1202",Windows/Sysmon,"T1059, T1202","T1202, T1059",N/A,T1202
81,PingCastle Binary Execution,HIGH,"Execution, Defense Evasion","T1059, T1202",Windows/Sysmon,"T1059, T1202","T1202, T1059",N/A,T1059


In [311]:
colums_sel = ['Rule','Priority','Technique','Mitre Technique','Name Mitre Technique','Mitre Tactic','Log Source Type']
UCMCatalog2024_Qradar_winsysmon = UCMCatalog2024_Qradar_winsysmon[colums_sel]
UCMCatalog2024_Qradar_winsysmon.tail(2)

,Rule,Priority,Technique,Mitre Technique,Name Mitre Technique,Mitre Tactic,Log Source Type
81,PingCastle Binary Execution,HIGH,T1202,"T1059, T1202",N/A,"Execution, Defense Evasion",Windows/Sysmon
81,PingCastle Binary Execution,HIGH,T1059,"T1059, T1202",N/A,"Execution, Defense Evasion",Windows/Sysmon


In [312]:
for indice, row in UCMCatalog2024_Qradar_winsysmon.iterrows():
    folder_name = row['Technique'].replace(' ', '')
    file_name = 'WindowsSysmon - '+row['Rule'] + ".csv"
    file_name = clean_names(file_name)
    folder_path = os.path.join(save_path_qradar, folder_name)
    csv_file = os.path.join(save_path_qradar, folder_name, file_name)
    os.makedirs(folder_path, exist_ok=True)
    temp_df = pd.DataFrame(row).T
    temp_df.columns = UCMCatalog2024_Qradar_winsysmon.columns
    temp_df.to_csv(csv_file, index=False, header=True, sep=';', quoting=csv.QUOTE_ALL)

*Atomics*

In [313]:
UCMCatalog2024_Qradar_atomics = pd.read_excel(path_UCMCatalog2024_Qradar,'Atomics')

UCMCatalog2024_Qradar_atomics['Mitre Technique'] = UCMCatalog2024_Qradar_atomics['Mitre Technique'].str.replace('\n', '', regex=False)

UCMCatalog2024_Qradar_atomics['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_atomics['Mitre Technique'].str.upper()
UCMCatalog2024_Qradar_atomics['techniques_from_raw'] = UCMCatalog2024_Qradar_atomics['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_atomics = UCMCatalog2024_Qradar_atomics.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_atomics['Mitre Tactic'] = UCMCatalog2024_Qradar_atomics['Mitre Tactic'].fillna('N/A')
UCMCatalog2024_Qradar_atomics['Name Mitre Technique'] = UCMCatalog2024_Qradar_atomics['Mitre Technique'].str.split('-').str[1]
UCMCatalog2024_Qradar_atomics['techniques_from_raw_explode'] = UCMCatalog2024_Qradar_atomics['techniques_from_raw'].str.split(',')
UCMCatalog2024_Qradar_atomics = UCMCatalog2024_Qradar_atomics.explode('techniques_from_raw_explode')
UCMCatalog2024_Qradar_atomics = UCMCatalog2024_Qradar_atomics[UCMCatalog2024_Qradar_atomics['techniques_from_raw']!='']
UCMCatalog2024_Qradar_atomics = UCMCatalog2024_Qradar_atomics.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_atomics.rename(columns={'techniques_from_raw_explode': 'Technique','TREND': 'Trend'}, inplace=True)
UCMCatalog2024_Qradar_atomics.head(2)

,Trend,Rule,Priority,Mitre Tactic,Mitre Technique,Log Source Type,MITRE_TECHNIQUE,techniques_from_raw,Name Mitre Technique,Technique
0,2024 - Ransomware,[AtomicRedTeam] - [InhibitSystemRecovery #1] ...,HIGH,Impact,T1490 - Inhibit System Recovery,Windows/Sysmon/Powershell,T1490 - INHIBIT SYSTEM RECOVERY,T1490,Inhibit System Recovery,T1490
1,2024 - Ransomware,[AtomicRedTeam] - [InhibitSystemRecovery #2] ...,HIGH,Impact,T1490 - Inhibit System Recovery,Windows/Sysmon/Powershell,T1490 - INHIBIT SYSTEM RECOVERY,T1490,Inhibit System Recovery,T1490


In [314]:
colums_sel = ['Trend','Rule','Priority','Technique','Mitre Technique','Name Mitre Technique','Mitre Tactic','Log Source Type']
UCMCatalog2024_Qradar_atomics = UCMCatalog2024_Qradar_atomics[colums_sel]
UCMCatalog2024_Qradar_atomics.tail(2)

,Trend,Rule,Priority,Technique,Mitre Technique,Name Mitre Technique,Mitre Tactic,Log Source Type
54,2024 - Threats,[AtomicRedTeam] - [Smashjacker #1] - AppInit D...,MED,T1546,T1546.010 - Event Triggered Execution:AppInit ...,Event Triggered Execution:AppInit DLLs,Privilege Escalation\nPersistence,Windows/Sysmon/Powershell
55,2024 - Threats,[AtomicRedTeam] - [Smashjacker #2] - Web Brows...,MED,T1176,T1176 - Browser Extensions,Browser Extensions,Persistence,Windows/Sysmon/Powershell


In [315]:
for indice, row in UCMCatalog2024_Qradar_atomics.iterrows():
    folder_name = row['Technique'].replace(' ', '')
    file_name = 'Atomics - '+row['Rule'] + ".csv"
    file_name = clean_names(file_name)
    folder_path = os.path.join(save_path_qradar, folder_name)
    csv_file = os.path.join(save_path_qradar, folder_name, file_name)
    os.makedirs(folder_path, exist_ok=True)
    temp_df = pd.DataFrame(row).T
    temp_df.columns = UCMCatalog2024_Qradar_atomics.columns
    temp_df.to_csv(csv_file, index=False, header=True, sep=';', quoting=csv.QUOTE_ALL)

*FW*

In [316]:
UCMCatalog2024_Qradar_fw = pd.read_excel(path_UCMCatalog2024_Qradar,'FW')

UCMCatalog2024_Qradar_fw['Mitre Technique'] = UCMCatalog2024_Qradar_fw['Mitre Technique'].str.replace('\n', '', regex=False)

UCMCatalog2024_Qradar_fw['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_fw['Mitre Technique'].str.upper()
UCMCatalog2024_Qradar_fw['techniques_from_raw'] = UCMCatalog2024_Qradar_fw['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_fw = UCMCatalog2024_Qradar_fw.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_fw['Mitre Tactic'] = UCMCatalog2024_Qradar_fw['Mitre Tactic'].fillna('N/A')
UCMCatalog2024_Qradar_fw['Name Mitre Technique'] = UCMCatalog2024_Qradar_fw['Mitre Technique'].str.split('-').str[1]
UCMCatalog2024_Qradar_fw['techniques_from_raw_explode'] = UCMCatalog2024_Qradar_fw['techniques_from_raw'].str.split(',')
UCMCatalog2024_Qradar_fw = UCMCatalog2024_Qradar_fw.explode('techniques_from_raw_explode')
UCMCatalog2024_Qradar_fw = UCMCatalog2024_Qradar_fw[UCMCatalog2024_Qradar_fw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_fw = UCMCatalog2024_Qradar_fw.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_fw.rename(columns={'techniques_from_raw_explode': 'Technique'}, inplace=True)
UCMCatalog2024_Qradar_fw.head(2)

,Rule,Priority,Mitre Tactic,Mitre Technique,MITRE_TECHNIQUE,techniques_from_raw,Name Mitre Technique,Technique
10,[CyberProof] - [Cisco ASA] - Login to SSH Inte...,Low,Initial Access,T1078 - Valid Accounts,T1078 - VALID ACCOUNTS,T1078,Valid Accounts,T1078
11,[CyberProof] - [Cisco ASA] - User Deleted,Medium,Defense Evasion,T1070 - Indicator Removal,T1070 - INDICATOR REMOVAL,T1070,Indicator Removal,T1070


In [317]:
colums_sel = ['Rule','Priority','Technique','Mitre Technique','Name Mitre Technique','Mitre Tactic']
UCMCatalog2024_Qradar_fw = UCMCatalog2024_Qradar_fw[colums_sel]
UCMCatalog2024_Qradar_fw.tail(2)

,Rule,Priority,Technique,Mitre Technique,Name Mitre Technique,Mitre Tactic
15,[CyberProof] - [Cisco ASA] - Group Policy Deleted,Medium,T1484,T1484.001 - Group Policy Modification,Group Policy Modification,Defense Evasion
16,[CyberProof] - [Cisco ASA] - User Locked,Medium,T1531,T1531 - Account Access Removal,Account Access Removal,Impact


In [318]:
for indice, row in UCMCatalog2024_Qradar_fw.iterrows():
    folder_name = row['Technique'].replace(' ', '')
    file_name = 'FW - '+row['Rule'] + ".csv"
    file_name = clean_names(file_name)
    folder_path = os.path.join(save_path_qradar, folder_name)
    csv_file = os.path.join(save_path_qradar, folder_name, file_name)
    os.makedirs(folder_path, exist_ok=True)
    temp_df = pd.DataFrame(row).T
    temp_df.columns = UCMCatalog2024_Qradar_fw.columns
    temp_df.to_csv(csv_file, index=False, header=True, sep=';', quoting=csv.QUOTE_ALL)

*AWS*

In [319]:
UCMCatalog2024_Qradar_aws = pd.read_excel(path_UCMCatalog2024_Qradar,'AWS')

UCMCatalog2024_Qradar_aws['Mitre Technique'] = UCMCatalog2024_Qradar_aws['Mitre Technique'].str.replace('\n', '', regex=False)

UCMCatalog2024_Qradar_aws['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_aws['Mitre Technique'].str.upper()
UCMCatalog2024_Qradar_aws['techniques_from_raw'] = UCMCatalog2024_Qradar_aws['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_aws = UCMCatalog2024_Qradar_aws.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_aws['Mitre Tactic'] = UCMCatalog2024_Qradar_aws['Mitre Tactic'].fillna('N/A')
UCMCatalog2024_Qradar_aws['Name Mitre Technique'] = UCMCatalog2024_Qradar_aws['Mitre Technique'].str.split('-').str[1]
UCMCatalog2024_Qradar_aws['techniques_from_raw_explode'] = UCMCatalog2024_Qradar_aws['techniques_from_raw'].str.split(',')
UCMCatalog2024_Qradar_aws = UCMCatalog2024_Qradar_aws.explode('techniques_from_raw_explode')
UCMCatalog2024_Qradar_aws = UCMCatalog2024_Qradar_aws[UCMCatalog2024_Qradar_aws['techniques_from_raw']!='']
UCMCatalog2024_Qradar_aws = UCMCatalog2024_Qradar_aws.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_aws.rename(columns={'techniques_from_raw_explode': 'Technique'}, inplace=True)
UCMCatalog2024_Qradar_aws.head(2)

,Rule,Priority,Mitre Tactic,Mitre Technique,Log Source Type,MITRE_TECHNIQUE,techniques_from_raw,Name Mitre Technique,Technique
1,[CyberProof] - [AWS] - Config Disabling Channe...,HIGH,Defense Evasion,T1562.001 - Impair Defenses: Disable or Modify...,AWS,T1562.001 - IMPAIR DEFENSES: DISABLE OR MODIFY...,"T1562.001, T1562",Impair Defenses: Disable or Modify Tools,T1562.001
1,[CyberProof] - [AWS] - Config Disabling Channe...,HIGH,Defense Evasion,T1562.001 - Impair Defenses: Disable or Modify...,AWS,T1562.001 - IMPAIR DEFENSES: DISABLE OR MODIFY...,"T1562.001, T1562",Impair Defenses: Disable or Modify Tools,T1562


In [320]:
colums_sel = ['Rule','Priority','Technique','Mitre Technique','Name Mitre Technique','Mitre Tactic','Log Source Type']
UCMCatalog2024_Qradar_aws = UCMCatalog2024_Qradar_aws[colums_sel]
UCMCatalog2024_Qradar_aws.tail(2)

,Rule,Priority,Technique,Mitre Technique,Name Mitre Technique,Mitre Tactic,Log Source Type
8,[CyberProof] - [AWS] - AWS GuardDuty Trusted I...,HIGH,T1562,T1562.001 - Impair Defenses: Disable or Modify...,Impair Defenses: Disable or Modify Tools,Defense Evasion,AWS
9,[CyberProof] - [AWS] - Identity Center Identit...,HIGH,T1556,T1556 - Modify Authentication Process,Modify Authentication Process,Persistence,AWS


In [321]:
for indice, row in UCMCatalog2024_Qradar_aws.iterrows():
    folder_name = row['Technique'].replace(' ', '')
    file_name = 'AWS - '+row['Rule'] + ".csv"
    file_name = clean_names(file_name)
    folder_path = os.path.join(save_path_qradar, folder_name)
    csv_file = os.path.join(save_path_qradar, folder_name, file_name)
    os.makedirs(folder_path, exist_ok=True)
    temp_df = pd.DataFrame(row).T
    temp_df.columns = UCMCatalog2024_Qradar_aws.columns
    temp_df.to_csv(csv_file, index=False, header=True, sep=';', quoting=csv.QUOTE_ALL)

#### TTP's UCM Catalog 2024 Splunk *from excel

In [322]:
path_UCMCatalog2024 = r'C:\Users\jelopez\Documents\CyberProof\python\check_0905\UCM Catalog 2024'

path_UCMCatalog2024_Splunk = path_UCMCatalog2024 + '/UCM Catalog 2024 [Splunk].xlsx'

save_path_splunk = os.path.join(save_path, 'UCM Catalog 2024 Splunk')

In [ ]:
create_output_folder(save_path_splunk)

In [323]:
UCMCatalog2024_splunk = pd.read_excel(path_UCMCatalog2024_Splunk)

UCMCatalog2024_splunk['search'] = UCMCatalog2024_splunk['search'].str.replace('\n', '', regex=False)

UCMCatalog2024_splunk['MITRE_TECHNIQUE'] = UCMCatalog2024_splunk['Mitre Technique'].str.upper()
UCMCatalog2024_splunk['DESCRIPTION'] = UCMCatalog2024_splunk['description'].str.upper()
UCMCatalog2024_splunk['SEARCH'] = UCMCatalog2024_splunk['search'].str.upper()
UCMCatalog2024_splunk['techniques_from_raw'] = UCMCatalog2024_splunk['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_splunk['techniques_from_description'] = UCMCatalog2024_splunk['DESCRIPTION'].apply(lambda x: find_techniques(x))
UCMCatalog2024_splunk['techniques_from_search'] = UCMCatalog2024_splunk['SEARCH'].apply(lambda x: find_techniques(x))
UCMCatalog2024_splunk = UCMCatalog2024_splunk.fillna('N/A')
UCMCatalog2024_splunk.head(2)

,title,description,search,severity,security_domain,MITRE Tactic,Mitre Technique,MITRE_TECHNIQUE,DESCRIPTION,SEARCH,techniques_from_raw,techniques_from_description,techniques_from_search
0,Access - Abstraction Non-Us Login - Rule,Abstractors are not allowed to work within Pat...,index=duo sourcetype!=okta:im p- user=p-* fact...,high,access,N/A,N/A,N/A,ABSTRACTORS ARE NOT ALLOWED TO WORK WITHIN PAT...,INDEX=DUO SOURCETYPE!=OKTA:IM P- USER=P-* FACT...,,,
1,Access - Credential Stuffing - Rule,Detects automated injection of stolen username...,| tstats summariesonly=t count as actioncount ...,low,access,N/A,N/A,N/A,DETECTS AUTOMATED INJECTION OF STOLEN USERNAME...,| TSTATS SUMMARIESONLY=T COUNT AS ACTIONCOUNT ...,,,


*raw*

In [324]:
UCMCatalog2024_splunk_raw = UCMCatalog2024_splunk
UCMCatalog2024_splunk_raw['Name Mitre Technique'] = UCMCatalog2024_splunk_raw['Mitre Technique'].str.split('-').str[1]
UCMCatalog2024_splunk_raw['techniques_from_raw_explode'] = UCMCatalog2024_splunk_raw['techniques_from_raw'].str.split(',')
UCMCatalog2024_splunk_raw = UCMCatalog2024_splunk_raw.explode('techniques_from_raw_explode')
UCMCatalog2024_splunk_raw = UCMCatalog2024_splunk_raw[UCMCatalog2024_splunk_raw['techniques_from_raw']!='']
UCMCatalog2024_splunk_raw = UCMCatalog2024_splunk_raw.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_splunk_raw.rename(columns={'title': 'Rule','description': 'Description','search': 'Search','severity': 'Severity','techniques_from_raw_explode': 'Technique','security_domain': 'Security domain','MITRE Tactic': 'Mitre Tactic'}, inplace=True)
UCMCatalog2024_splunk_raw['Search'] = UCMCatalog2024_splunk_raw['Search'].str.replace('\n', '', regex=False)
colums_sel = ['Rule','Description','Severity','Technique','Mitre Technique','Name Mitre Technique','Mitre Tactic','Security domain','Search']
UCMCatalog2024_splunk_raw = UCMCatalog2024_splunk_raw[colums_sel]
UCMCatalog2024_splunk_raw.tail(2)

,Rule,Description,Severity,Technique,Mitre Technique,Name Mitre Technique,Mitre Tactic,Security domain,Search
2363,CyberProof - Ransomware - Clop Ransomware Know...,This detection is to identify the common servi...,N/A,T1543,"{""analytic_story"":[""Clop Ransomware""],""cis20"":...",NaN,N/A,N/A,index=windows sourcetype=WinEventLog eventtype...
2364,CyberProof - Ransomware - Known Services Kille...,This search detects a suspicioous termination ...,N/A,T1490,"{""analytic_story"":[""Ransomware"",""BlackMatter R...",NaN,N/A,N/A,index=windows sourcetype=WinEventLog eventtype...


In [325]:
UCMCatalog2024_splunk_description = UCMCatalog2024_splunk
UCMCatalog2024_splunk_description['Name Mitre Technique'] = UCMCatalog2024_splunk_description['Mitre Technique'].str.split('-').str[1]
UCMCatalog2024_splunk_description['techniques_from_description_explode'] = UCMCatalog2024_splunk_description['techniques_from_description'].str.split(',')
UCMCatalog2024_splunk_description = UCMCatalog2024_splunk_description.explode('techniques_from_description_explode')
UCMCatalog2024_splunk_description = UCMCatalog2024_splunk_description[UCMCatalog2024_splunk_description['techniques_from_description']!='']
UCMCatalog2024_splunk_description = UCMCatalog2024_splunk_description.dropna(subset=['techniques_from_description'])
UCMCatalog2024_splunk_description.rename(columns={'title': 'Rule','description': 'Description','search': 'Search','severity': 'Severity','techniques_from_description_explode': 'Technique','security_domain': 'Security domain','MITRE Tactic': 'Mitre Tactic'}, inplace=True)
UCMCatalog2024_splunk_description['Search'] = UCMCatalog2024_splunk_description['Search'].str.replace('\n', '', regex=False)
colums_sel = ['Rule','Description','Severity','Technique','Mitre Technique','Name Mitre Technique','Mitre Tactic','Security domain','Search']
UCMCatalog2024_splunk_description = UCMCatalog2024_splunk_description[colums_sel]
UCMCatalog2024_splunk_description.tail(2)

,Rule,Description,Severity,Technique,Mitre Technique,Name Mitre Technique,Mitre Tactic,Security domain,Search


In [326]:
UCMCatalog2024_splunk_search = UCMCatalog2024_splunk
UCMCatalog2024_splunk_search['Name Mitre Technique'] = UCMCatalog2024_splunk_search['Mitre Technique'].str.split('-').str[1]
UCMCatalog2024_splunk_search['techniques_from_search_explode'] = UCMCatalog2024_splunk_search['techniques_from_search'].str.split(',')
UCMCatalog2024_splunk_search = UCMCatalog2024_splunk_search.explode('techniques_from_search_explode')
UCMCatalog2024_splunk_search = UCMCatalog2024_splunk_search[UCMCatalog2024_splunk_search['techniques_from_search']!='']
UCMCatalog2024_splunk_search = UCMCatalog2024_splunk_search.dropna(subset=['techniques_from_search'])
UCMCatalog2024_splunk_search.rename(columns={'title': 'Rule','description': 'Description','search': 'Search','severity': 'Severity','techniques_from_search_explode': 'Technique','security_domain': 'Security domain','MITRE Tactic': 'Mitre Tactic'}, inplace=True)
UCMCatalog2024_splunk_search['Search'] = UCMCatalog2024_splunk_search['Search'].str.replace('\n', '', regex=False)
colums_sel = ['Rule','Description','Severity','Technique','Mitre Technique','Name Mitre Technique','Mitre Tactic','Security domain','Search']
UCMCatalog2024_splunk_search = UCMCatalog2024_splunk_search[colums_sel]
UCMCatalog2024_splunk_search.tail(2)

,Rule,Description,Severity,Technique,Mitre Technique,Name Mitre Technique,Mitre Tactic,Security domain,Search


In [327]:
UCMCatalog2024_Splunk_union = pd.concat([UCMCatalog2024_splunk_raw, UCMCatalog2024_splunk_description, UCMCatalog2024_splunk_search])
UCMCatalog2024_Splunk_union.shape

(105, 9)

In [328]:
for indice, row in UCMCatalog2024_Splunk_union.iterrows():
    folder_name = row['Technique'].replace(' ', '')
    file_name = row['Rule'] + ".csv"
    file_name = clean_names(file_name)
    folder_path = os.path.join(save_path_splunk, folder_name)
    csv_file = os.path.join(save_path_splunk, folder_name, file_name)
    os.makedirs(folder_path, exist_ok=True)
    temp_df = pd.DataFrame(row).T
    temp_df.columns = UCMCatalog2024_Splunk_union.columns
    temp_df.to_csv(csv_file, index=False, header=True, sep=';', quoting=csv.QUOTE_ALL)

#### TTP's Sigma HQ 1

<h5 style="background-color: #E11717; color: black;">
    Importante: No ejecutar por solicitud de Seguridad
</h5>

In [16]:
# zip_sigma_hq1 = 'https://github.com/SigmaHQ/sigma/archive/refs/heads/master.zip'
save_path_sigma_hq1 = os.path.join(save_path, 'sigmaHQ1')

In [17]:
create_output_folder(save_path_sigma_hq1)

In [18]:
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-compliance', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-dfir', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-emerging-threats', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-placeholder', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-threat-hunting', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-placeholder', temp_path)

Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.


In [19]:
files = getListOfFilesSub(temp_path+r'\sigma-master')
files[:3]

['C:\\Users\\jelopez\\Downloads\\sigma-master\\rules\\application\\django\\appframework_django_exceptions.yml',
 'C:\\Users\\jelopez\\Downloads\\sigma-master\\rules\\application\\jvm\\java_jndi_injection_exploitation_attempt.yml',
 'C:\\Users\\jelopez\\Downloads\\sigma-master\\rules\\application\\jvm\\java_local_file_read.yml']

In [20]:
# Recorrer la lista de archivos
num_yamls = []
has_label_tags = []
has_label_with_content_tags = []
has_label_without_content_tags = []
no_label_tags = []
notTTPinTags = []
notTTPinTags_with_TTP_in_description = []
notTTPinTags_without_description = []
error_load = []

for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yaml' or extension == '.yml':
        with open(item) as file:
            num_yamls.append(item)
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
                document = yaml.full_load(file)
                if 'tags' in document:
                    # Si encontramos la etiqueta 'tags' en el yaml lo guardamos como una variable 
                    tags = document.get('tags', [])
                    has_label_tags.append(item)
                    # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                    if not isinstance(tags, type(None)):
                        has_label_with_content_tags.append(item)
                        # Convertimos en lista el contenido de la etiqueta
                        #tags = [tags]
                        tags = [tag.upper() for tag in tags if isinstance(tag, str)]
                        # Recorremos la lista de items contenidos en tags y contamos las ttp encontradas ya que si no encontramos ninguna, pese haber texto en la etiqueta
                        # deberemos mover el fichero a la carpeta T0000
                        ttp_counter = 0
                        for tag in range(len(tags)):
                            # Matcheamos el item de la lista tags con nuestra lista de ttps 
                            ttp = find_techniques_in_list(tags[tag],techniques_enterprise)
                            # Y si es un string lo que tenemos, entonces guardamos en la carpeta de la la ttp correspondiente
                            # Ojo,¿que pasa si la lista tiene un string pero no una ttp?
                            if isinstance(ttp, str):
                                ttp_counter += 1
                                copy_file_to_path(item, save_path_sigma_hq1, ttp)
                        if ttp_counter == 0:
                            raise NotTTPatTags('.yaml sin TTP encontrada en los tags')
                
                    else:
                        # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
                        has_label_without_content_tags.append(item)
                        if 'description' in document:
                            description = [document.get('description', [])]
                            # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                            for tag in range(len(description)):
                                ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                                if not isinstance(ttp, type(None)):
                                    if not os.path.exists(os.path.join(save_path_sigma_hq1, ttp)):
                                        os.makedirs(os.path.join(save_path_sigma_hq1, ttp))
                                else:
                                    ttp='T0000'
                                copy_file_to_path(item, save_path_sigma_hq1, ttp)
                else:
                    no_label_tags.append(item)
                    # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'tags' no exista.
                    if 'description' in document:
                        description = [document.get('description', [])]
                        # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                        for tag in range(len(description)):
                            ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                            if not isinstance(ttp, type(None)):
                                if not os.path.exists(os.path.join(save_path_sigma_hq1, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq1, ttp))
                            else:
                                ttp='T0000'
                            copy_file_to_path(item, save_path_sigma_hq1, ttp)
                    else:
                        copy_file_to_path(item, save_path_sigma_hq1, 'T0000')
            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except yaml.YAMLError as e:
                print(f"Error al cargar el archivo YAML: {e}")
                error_load.append(item)
                copy_file_to_path(item, save_path_sigma_hq1, 'T0000')
            
            # Con esta excepción llegamos si hemos recorrido la lista de tags y no hemos encontrado ninguna TTP, por lo que revisamos la etiqueta dando lugar a 3 casuísticas,
            # la primera que encuentre dicha etiqueta, en cuyo caso se revisa el contenido, si dentro de este se encuentra alguna TTP se mandara a la carpeta correspondiente. En caso de no haber coincidencia, al igual que ocurrirá si no hay etiqueta 'description' el fichero será copiado a T000.
            except NotTTPatTags as e:
                print(f"{e}")
                notTTPinTags.append(item)
                if 'description' in document:
                    description = [document.get('description', [])]
                    # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                    for tag in range(len(description)):
                        ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                        if not isinstance(ttp, type(None)):
                            notTTPinTags_with_TTP_in_description.append(item)
                            pass
                        else:
                            ttp='T0000'
                        copy_file_to_path(item, save_path_sigma_hq1, ttp)
                else:
                    notTTPinTags_without_description.append(item)
                    copy_file_to_path(item, save_path_sigma_hq1, 'T0000')

Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1498
Copiado correctamente a la carpeta de la TTP: T1070
Copiado correctamente a la carpeta de la TTP: T1609
Copiado correctamente a la carpeta de la TTP: T1611
Copiado correctamente a la carpeta de la TTP: T1036.005
Copiado correctamente a la carpeta de la TTP: T1611
Copiado correctamente a la carpeta de la TTP: T1069.003
Copiado correctamente a la carpeta de la TTP: T1087.004
Copiado correctamente a la carpeta de la TTP: T1552.007
Copiado correctamente a la carpeta de la TTP: T1136
Copiado correctamente a la carpeta de la TTP: T1609
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1

In [30]:
print('Reglas totales para asignar: '+str(len(num_yamls)))
rules_unique = get_unique_items(num_yamls)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 3320
Reglas únicas para asignar: 3320


In [26]:
print('Tienen informada la etiqueta "tags": '+str(len(has_label_tags)))
print('Tienen información en la etiqueta "tags": '+str(len(has_label_with_content_tags)))
print('NO tienen información en la etiqueta "tags": '+str(len(has_label_without_content_tags)))

Tienen informada la etiqueta "tags": 3318
Tienen información en la etiqueta "tags": 3318
NO tienen información en la etiqueta "tags": 0


In [27]:
print('NO tienen la etiqueta "tags": '+str(len(no_label_tags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags": '+str(len(notTTPinTags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": '+str(len(notTTPinTags_without_description)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": '+str(len(notTTPinTags_with_TTP_in_description)))

NO tienen la etiqueta "tags": 2
NO se ha encontrado TTP alguna en la etiqueta "tags": 448
NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": 0
NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": 1


In [28]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_sigma_hq1))))
print('Reglas sin TTP asignada (T0000): '+str(len(getListOfFilesSub(os.path.join(save_path_sigma_hq1, 'T0000')))))
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_sigma_hq1)) - len(getListOfFilesSub(os.path.join(save_path_sigma_hq1, 'T0000')))))
print('Reglas asignadas a 2 o más TTPs: '+str((len(getListOfFilesSub(save_path_sigma_hq1)) - len(getListOfFilesSub(os.path.join(save_path_sigma_hq1, 'T0000'))))-len(rules_unique)))

Número de TTPs únicas identificadas (puede incluir T0000): 376
Reglas sin TTP asignada (T0000): 449
Reglas asignadas a TTP: 3808
Reglas asignadas a 2 o más TTPs: 488


#### TTP's Sigma HQ 2

<h5 style="background-color: #E11717; color: black;">
    Importante: No ejecutar por solicitud de Seguridad
</h5>

In [17]:
# zip_sigma_hq2 = 'https://github.com/mdecrevoisier/SIGMA-detection-rules/archive/refs/heads/main.zip'
save_path_sigma_hq2 = os.path.join(save_path, 'sigmaHQ2')
unzip_folder = r'\SIGMA-detection-rules-main'
techniques_enterprise[:3]


['T1059.010', 'T1564.012', 'T1027.013']

In [18]:
create_output_folder(save_path_sigma_hq2)

In [19]:
download_unzip(zip_sigma_hq2, temp_path)

In [20]:
files = getListOfFilesSub(temp_path+unzip_folder)
files[:3]

['C:\\Users\\jelopez\\Downloads\\SIGMA-detection-rules-main\\.gitignore',
 'C:\\Users\\jelopez\\Downloads\\SIGMA-detection-rules-main\\cloud-azure\\azure-active-directory-new federated trust added.yaml',
 'C:\\Users\\jelopez\\Downloads\\SIGMA-detection-rules-main\\cloud-azure\\azure-exchange-email forwarding to external domain.yaml']

In [21]:
# Recorrer la lista de archivos
num_yamls = []
has_label_tags = []
has_label_with_content_tags = []
has_label_without_content_tags = []
no_label_tags = []
notTTPinTags = []
notTTPinTags_with_TTP_in_description = []
notTTPinTags_without_description = []
error_load = []

for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yaml' or extension == '.yml':
        with open(item) as file:
            num_yamls.append(item)
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
                document = yaml.full_load(file)
                if 'tags' in document:
                    # Si encontramos la etiqueta 'tags' en el yaml lo guardamos como una variable 
                    tags = document.get('tags', [])
                    has_label_tags.append(item)
                    # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                    if not isinstance(tags, type(None)):
                        has_label_with_content_tags.append(item)
                        # Convertimos en lista el contenido de la etiqueta
                        #tags = [tags]
                        tags = [tag.upper() for tag in tags if isinstance(tag, str)]
                        # Recorremos la lista de items contenidos en tags y contamos las ttp encontradas ya que si no encontramos ninguna, pese haber texto en la etiqueta
                        # deberemos mover el fichero a la carpeta T0000
                        ttp_counter = 0
                        for tag in range(len(tags)):
                            # Matcheamos el item de la lista tags con nuestra lista de ttps 
                            ttp = find_techniques_in_list(tags[tag],techniques_enterprise)
                            # Y si es un string lo que tenemos, entonces guardamos en la carpeta de la la ttp correspondiente
                            # Ojo,¿que pasa si la lista tiene un string pero no una ttp?
                            if isinstance(ttp, str):
                                ttp_counter += 1
                                copy_file_to_path(item, save_path_sigma_hq2, ttp)
                        if ttp_counter == 0:
                            raise NotTTPatTags('.yaml sin TTP encontrada en los tags')
                
                    else:
                        # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
                        has_label_without_content_tags.append(item)
                        if 'description' in document:
                            description = [document.get('description', [])]
                            # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                            for tag in range(len(description)):
                                ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                                if not isinstance(ttp, type(None)):
                                    if not os.path.exists(os.path.join(save_path_sigma_hq2, ttp)):
                                        os.makedirs(os.path.join(save_path_sigma_hq2, ttp))
                                else:
                                    ttp='T0000'
                                copy_file_to_path(item, save_path_sigma_hq2, ttp)
                else:
                    no_label_tags.append(item)
                    # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'tags' no exista.
                    if 'description' in document:
                        description = [document.get('description', [])]
                        # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                        for tag in range(len(description)):
                            ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                            if not isinstance(ttp, type(None)):
                                if not os.path.exists(os.path.join(save_path_sigma_hq2, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq2, ttp))
                            else:
                                ttp='T0000'
                            copy_file_to_path(item, save_path_sigma_hq2, ttp)
                    else:
                        copy_file_to_path(item, save_path_sigma_hq2, 'T0000')
            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except yaml.YAMLError as e:
                print(f"Error al cargar el archivo YAML: {e}")
                error_load.append(item)
                copy_file_to_path(item, save_path_sigma_hq2, 'T0000')
            
            # Con esta excepción llegamos si hemos recorrido la lista de tags y no hemos encontrado ninguna TTP, por lo que revisamos la etiqueta dando lugar a 3 casuísticas,
            # la primera que encuentre dicha etiqueta, en cuyo caso se revisa el contenido, si dentro de este se encuentra alguna TTP se mandara a la carpeta correspondiente. En caso de no haber coincidencia, al igual que ocurrirá si no hay etiqueta 'description' el fichero será copiado a T000.
            except NotTTPatTags as e:
                print(f"{e}")
                notTTPinTags.append(item)
                if 'description' in document:
                    description = [document.get('description', [])]
                    # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                    for tag in range(len(description)):
                        ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                        if not isinstance(ttp, type(None)):
                            notTTPinTags_with_TTP_in_description.append(item)
                            pass
                        else:
                            ttp='T0000'
                        copy_file_to_path(item, save_path_sigma_hq2, ttp)
                else:
                    notTTPinTags_without_description.append(item)
                    copy_file_to_path(item, save_path_sigma_hq2, 'T0000')

Copiado correctamente a la carpeta de la TTP: T1484.002
Copiado correctamente a la carpeta de la TTP: T1114.003
Copiado correctamente a la carpeta de la TTP: T1114
Copiado correctamente a la carpeta de la TTP: T1566
Copiado correctamente a la carpeta de la TTP: T1114
Copiado correctamente a la carpeta de la TTP: T1566
Copiado correctamente a la carpeta de la TTP: T1562.006
Copiado correctamente a la carpeta de la TTP: T1546
Copiado correctamente a la carpeta de la TTP: T1110.001
Copiado correctamente a la carpeta de la TTP: T1110.003
Copiado correctamente a la carpeta de la TTP: T1136
Copiado correctamente a la carpeta de la TTP: T1098
Copiado correctamente a la carpeta de la TTP: T1136
Copiado correctamente a la carpeta de la TTP: T1068
Copiado correctamente a la carpeta de la TTP: T1222.001
Copiado correctamente a la carpeta de la TTP: T1098
Copiado correctamente a la carpeta de la TTP: T1036
Copiado correctamente a la carpeta de la TTP: T1068
Copiado correctamente a la carpeta de la

In [47]:
print('Reglas totales para asignar: '+str(len(yamls)))
rules_unique = get_unique_items(yamls)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 348
Reglas únicas para asignar: 348


In [48]:
print('Tienen informada la etiqueta "tags": '+str(len(has_label_tags)))
print('Tienen información en la etiqueta "tags": '+str(len(has_label_with_content_tags)))
print('NO tienen información en la etiqueta "tags": '+str(len(has_label_without_content_tags)))

Tienen informada la etiqueta "tags": 347
Tienen información en la etiqueta "tags": 347
NO tienen información en la etiqueta "tags": 0


In [52]:
print('NO tienen la etiqueta "tags": '+str(len(no_label_tags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags": '+str(len(notTTPinTags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": '+str(len(notTTPinTags_without_description)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": '+str(len(notTTPinTags_with_TTP_in_description)))

NO tienen la etiqueta "tags": 0
NO se ha encontrado TTP alguna en la etiqueta "tags": 4
NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": 0
NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": 0


In [84]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_sigma_hq2))))
print('Reglas sin TTP asignada (T0000): '+str(len(getListOfFilesSub(os.path.join(save_path_sigma_hq2, 'T0000')))))
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_sigma_hq2)) - len(getListOfFilesSub(os.path.join(save_path_sigma_hq2, 'T0000')))))
print('Reglas asignadas a 2 o más TTPs: '+str((len(getListOfFilesSub(save_path_sigma_hq2)) - len(getListOfFilesSub(os.path.join(save_path_sigma_hq2, 'T0000'))))-len(rules_unique)))


Número de TTPs únicas identificadas (puede incluir T0000): 111
Reglas sin TTP asignada (T0000): 5
Reglas asignadas a TTP: 410
Reglas asignadas a 2 o más TTPs: 62


#### TTP's Sigma HQ 3

In [11]:
zip_sigma_hq3 = 'https://github.com/joesecurity/sigma-rules/archive/refs/heads/master.zip'
save_path_sigma_hq3 = os.path.join(save_path, 'sigmaHQ3')
techniques_enterprise = techniques()
techniques_enterprise = sorted(techniques_enterprise, key=len, reverse=True)


[taxii2client.v20] [WARNING ] [2024-05-23 11:38:42,264] TAXII Server Response did not include 'Content-Range' header - results could be incomplete.
[taxii2client.v20] [WARNING ] [2024-05-23 11:38:42,296] TAXII Server Response with different amount of objects! Setting per_request=780


In [568]:
create_output_folder(save_path_sigma_hq3)

In [569]:
download_unzip(zip_sigma_hq3, temp_path)

In [12]:
files = getListOfFilesSub(temp_path+r'\sigma-rules-master')
files[0:3]

['C:\\Users\\jelopez\\Downloads\\sigma-rules-master\\images\\sigma.png',
 'C:\\Users\\jelopez\\Downloads\\sigma-rules-master\\LICENSE',
 'C:\\Users\\jelopez\\Downloads\\sigma-rules-master\\README.md']

In [41]:
# Recorrer la lista de archivos
num_yamls = []
has_label_mitreattack = []
has_label_with_content_mitreattack = []
has_label_without_content_mitreattack = []
no_label_mitreattack = []
error_load = []

for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yaml' or extension == '.yml':
        with open(item) as file:
            TTP_list = [] #?
            num_yamls.append(item)
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
                document = yaml.full_load(file)
                if 'mitreattack' in document:
                    # Si encontramos el tag 'mitreattack' en el yaml lo guardamos como una variable 
                    mitreattacks = document.get('mitreattack', [])
                    has_label_mitreattack.append(item)
                    # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                    if not isinstance(mitreattacks, type(None)):
                        has_label_with_content_mitreattack.append(item)
                        # Convertimos en lista el contenido de la etiqueta
                        mitreattacks = [mitreattacks]
                        if isinstance(mitreattacks, list):
                            # Chequeamos que se haya podido generar correctamente la lista y la vamos recorriendo añadiendo las ttps encontradas en el listado de Mitre
                            # Guardamos el fichero en la carpeta correspondiente.
                            for tag in range(len(mitreattacks)):
                                ttp = find_techniques_in_list(mitreattacks[tag],techniques_enterprise)
                                if not os.path.exists(os.path.join(save_path_sigma_hq3, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq3, ttp))
                                shutil.copy(item,os.path.join(save_path_sigma_hq3, ttp))
                                print('Copiado correctamente a la carpeta de la TTP: '+ ttp)   
                    else:
                        # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
                        has_label_without_content_mitreattack.append(item)
                        if 'description' in document:
                            description = [document.get('description', [])]
                            # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                            for tag in range(len(description)):
                                ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                                if not isinstance(ttp, type(None)):
                                    if not os.path.exists(os.path.join(save_path_sigma_hq3, ttp)):
                                        os.makedirs(os.path.join(save_path_sigma_hq3, ttp))
                                else:
                                    ttp='T0000'
                                if not os.path.exists(os.path.join(save_path_sigma_hq3, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq3, 'T0000'))
                                shutil.copy(item,os.path.join(save_path_sigma_hq3, ttp))
                                print('Copiado correctamente a la carpeta de la TTP: '+ ttp)
                else:
                    no_label_mitreattack.append(item)
                    # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'mitreattack' no exista
                    if 'description' in document:
                        description = [document.get('description', [])]
                        # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                        for tag in range(len(description)):
                            ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                            if not isinstance(ttp, type(None)):
                                if not os.path.exists(os.path.join(save_path_sigma_hq3, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq3, ttp))
                            else:
                                ttp='T0000'
                            if not os.path.exists(os.path.join(save_path_sigma_hq3, ttp)):
                                os.makedirs(os.path.join(save_path_sigma_hq3, 'T0000'))
                            shutil.copy(item,os.path.join(save_path_sigma_hq3, ttp))
                            print('Copiado correctamente a la carpeta de la TTP: '+ ttp)
            except yaml.YAMLError as e:
                print(f"Error al cargar el archivo YAML: {e}")
                error_load.append(item)
                if not os.path.exists(os.path.join(save_path_sigma_hq3, 'T0000')):
                    os.makedirs(os.path.join(save_path_sigma_hq3, 'T0000'))
                shutil.copy(item,os.path.join(save_path_sigma_hq3, 'T0000'))
                print('Copiado correctamente correctamente a la carpeta T0000')


Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T1497
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T1574.002
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T1490
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado 

In [32]:
type(mitreattacks[0])

NoneType

In [24]:
num_yamls = []
has_label_mitreattack = []
has_label_with_content_mitreattack = []
has_label_without_content_mitreattack = []
no_label_mitreattack = []
error_load = []

7

In [31]:
len(no_label_mitreattack)

14

In [42]:
len(getListOfFilesSub(save_path_sigma_hq3))

118

In [43]:
len(getListOfFilesSub(os.path.join(save_path_sigma_hq3, 'T0000')))

105

#### TTP's Sigma HQ 4

<h5 style="background-color: #E11717; color: black;">
    Importante: No ejecutar por solicitud de Seguridad
</h5>

In [32]:
# zip_sigma_hq4 = 'https://github.com/SigmaHQ/sigma/archive/refs/heads/master.zip'
save_path_sigma_hq4 = os.path.join(save_path, 'sigmaHQ4')
unzip_folder = r'sigma-master/rules/'

In [36]:
download_folder_unzip(zip_sigma_hq4, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [37]:
files = getListOfFilesSub(os.path.join(temp_path, unzip_folder))
len(files)

2889

In [58]:
create_output_folder(save_path_sigma_hq4)

In [38]:
# Recorrer la lista de archivos
num_yamls = []
has_label_tags = []
has_label_with_content_tags = []
has_label_without_content_tags = []
no_label_tags = []
notTTPinTags = []
notTTPinTags_with_TTP_in_description = []
notTTPinTags_without_description = []
error_load = []

for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yaml' or extension == '.yml':
        with open(item) as file:
            num_yamls.append(item)
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
                document = yaml.full_load(file)
                if 'tags' in document:
                    # Si encontramos la etiqueta 'tags' en el yaml lo guardamos como una variable 
                    tags = document.get('tags', [])
                    has_label_tags.append(item)
                    # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                    if not isinstance(tags, type(None)):
                        has_label_with_content_tags.append(item)
                        # Convertimos en lista el contenido de la etiqueta
                        #tags = [tags]
                        tags = [tag.upper() for tag in tags if isinstance(tag, str)]
                        # Recorremos la lista de items contenidos en tags y contamos las ttp encontradas ya que si no encontramos ninguna, pese haber texto en la etiqueta
                        # deberemos mover el fichero a la carpeta T0000
                        ttp_counter = 0
                        for tag in range(len(tags)):
                            # Matcheamos el item de la lista tags con nuestra lista de ttps 
                            ttp = find_techniques_in_list(tags[tag],techniques_enterprise)
                            # Y si es un string lo que tenemos, entonces guardamos en la carpeta de la la ttp correspondiente
                            # Ojo,¿que pasa si la lista tiene un string pero no una ttp?
                            if isinstance(ttp, str):
                                ttp_counter += 1
                                copy_file_to_path(item, save_path_sigma_hq4, ttp)
                        if ttp_counter == 0:
                            raise NotTTPatTags('.yaml sin TTP encontrada en los tags')
                
                    else:
                        # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
                        has_label_without_content_tags.append(item)
                        if 'description' in document:
                            description = [document.get('description', [])]
                            # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                            for tag in range(len(description)):
                                ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                                if not isinstance(ttp, type(None)):
                                    if not os.path.exists(os.path.join(save_path_sigma_hq4, ttp)):
                                        os.makedirs(os.path.join(save_path_sigma_hq4, ttp))
                                else:
                                    ttp='T0000'
                                copy_file_to_path(item, save_path_sigma_hq4, ttp)
                else:
                    no_label_tags.append(item)
                    # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'tags' no exista.
                    if 'description' in document:
                        description = [document.get('description', [])]
                        # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                        for tag in range(len(description)):
                            ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                            if not isinstance(ttp, type(None)):
                                if not os.path.exists(os.path.join(save_path_sigma_hq4, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq4, ttp))
                            else:
                                ttp='T0000'
                            copy_file_to_path(item, save_path_sigma_hq4, ttp)
                    else:
                        copy_file_to_path(item, save_path_sigma_hq4, 'T0000')
            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except yaml.YAMLError as e:
                print(f"Error al cargar el archivo YAML: {e}")
                error_load.append(item)
                copy_file_to_path(item, save_path_sigma_hq4, 'T0000')
            
            # Con esta excepción llegamos si hemos recorrido la lista de tags y no hemos encontrado ninguna TTP, por lo que revisamos la etiqueta dando lugar a 3 casuísticas,
            # la primera que encuentre dicha etiqueta, en cuyo caso se revisa el contenido, si dentro de este se encuentra alguna TTP se mandara a la carpeta correspondiente. En caso de no haber coincidencia, al igual que ocurrirá si no hay etiqueta 'description' el fichero será copiado a T000.
            except NotTTPatTags as e:
                print(f"{e}")
                notTTPinTags.append(item)
                if 'description' in document:
                    description = [document.get('description', [])]
                    # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                    for tag in range(len(description)):
                        ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                        if not isinstance(ttp, type(None)):
                            notTTPinTags_with_TTP_in_description.append(item)
                            pass
                        else:
                            ttp='T0000'
                        copy_file_to_path(item, save_path_sigma_hq4, ttp)
                else:
                    notTTPinTags_without_description.append(item)
                    copy_file_to_path(item, save_path_sigma_hq4, 'T0000')

Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1498
Copiado correctamente a la carpeta de la TTP: T1070
Copiado correctamente a la carpeta de la TTP: T1609
Copiado correctamente a la carpeta de la TTP: T1611
Copiado correctamente a la carpeta de la TTP: T1036.005
Copiado correctamente a la carpeta de la TTP: T1611
Copiado correctamente a la carpeta de la TTP: T1069.003
Copiado correctamente a la carpeta de la TTP: T1087.004
Copiado correctamente a la carpeta de la TTP: T1552.007
Copiado correctamente a la carpeta de la TTP: T1136
Copiado correctamente a la carpeta de la TTP: T1609
Copiado correctamente a la carpeta de la TTP: T1190
Copiado correctamente a la carpeta de la TTP: T1

In [39]:
print('Reglas totales para asignar: '+str(len(num_yamls)))
rules_unique = get_unique_items(num_yamls)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 2888
Reglas únicas para asignar: 2888


In [40]:
print('Tienen informada la etiqueta "tags": '+str(len(has_label_tags)))
print('Tienen información en la etiqueta "tags": '+str(len(has_label_with_content_tags)))
print('NO tienen información en la etiqueta "tags": '+str(len(has_label_without_content_tags)))

Tienen informada la etiqueta "tags": 2887
Tienen información en la etiqueta "tags": 2887
NO tienen información en la etiqueta "tags": 0


In [41]:
print('NO tienen la etiqueta "tags": '+str(len(no_label_tags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags": '+str(len(notTTPinTags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": '+str(len(notTTPinTags_without_description)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": '+str(len(notTTPinTags_with_TTP_in_description)))

NO tienen la etiqueta "tags": 1
NO se ha encontrado TTP alguna en la etiqueta "tags": 338
NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": 0
NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": 1


In [42]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_sigma_hq4))))
print('Reglas sin TTP asignada (T0000): '+str(len(getListOfFilesSub(os.path.join(save_path_sigma_hq4, 'T0000')))))
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_sigma_hq4)) - len(getListOfFilesSub(os.path.join(save_path_sigma_hq4, 'T0000')))))
print('Reglas asignadas a 2 o más TTPs: '+str((len(getListOfFilesSub(save_path_sigma_hq4)) - len(getListOfFilesSub(os.path.join(save_path_sigma_hq4, 'T0000'))))-len(rules_unique)))

Número de TTPs únicas identificadas (puede incluir T0000): 372
Reglas sin TTP asignada (T0000): 338
Reglas asignadas a TTP: 3377
Reglas asignadas a 2 o más TTPs: 489


#### TTP's Sigma HQ 5

In [46]:
zip_sigma_hq5 = 'https://github.com/socprime/socprime_sigma/archive/refs/heads/master.zip'
save_path_sigma_hq5 = os.path.join(save_path, 'sigmaHQ5')
unzip_folder = r'socprime_sigma-master'

In [44]:
download_unzip(zip_sigma_hq5, temp_path)

In [47]:
files = getListOfFilesSub(os.path.join(temp_path, unzip_folder))
len(files)

37

In [48]:
create_output_folder(save_path_sigma_hq5)

In [59]:
num_yamls = []
has_label_tags = []
has_label_with_content_tags = []
has_label_without_content_tags = []
no_label_tags = []
notTTPinTags = []
notTTPinTags_with_TTP_in_description = []
notTTPinTags_without_description = []
error_load = []

ttps_in_tags_not_in_techniques_enterprise = []
ttps_in_tags_not_in_techniques_enterprise_items = []

for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yaml' or extension == '.yml':
        with open(item) as file:
            num_yamls.append(item)
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
                document = yaml.full_load(file)
                if 'tags' in document:
                    # Si encontramos la etiqueta 'tags' en el yaml lo guardamos como una variable 
                    tags = document.get('tags', [])
                    has_label_tags.append(item)
                    # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                    if not isinstance(tags, type(None)):
                        has_label_with_content_tags.append(item)
                        # Convertimos en lista el contenido de la etiqueta
                        #tags = [tags]
                        tags = [tag.upper() for tag in tags if isinstance(tag, str)]
                        # Recorremos la lista de items contenidos en tags y contamos las ttp encontradas ya que si no encontramos ninguna, pese haber texto en la etiqueta
                        # deberemos mover el fichero a la carpeta T0000
                        ttp_counter = 0
                        for tag in range(len(tags)):
                            # Matcheamos el item de la lista tags con nuestra lista de ttps 
                            ttp = find_techniques_in_list(tags[tag],techniques_enterprise)
                            # He encontrado en sigma HQ 5 tags que contienen técnicas que no están en el listado de techniques_enterprise, vamos a almacenarlas para reportarlas.
                            if '.T0' in tags[tag] or '.T1' in tags[tag]:
                                ttps_in_tags_not_in_techniques_enterprise.append(tags[tag])
                                ttps_in_tags_not_in_techniques_enterprise_items.append(item)
                            # Y si es un string lo que tenemos, entonces guardamos en la carpeta de la la ttp correspondiente
                            if isinstance(ttp, str):
                                ttp_counter += 1
                                copy_file_to_path(item, save_path_sigma_hq5, ttp)
                        if ttp_counter == 0:
                            raise NotTTPatTags('.yaml sin TTP encontrada en los tags')
                
                    else:
                        # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
                        has_label_without_content_tags.append(item)
                        if 'description' in document:
                            description = [document.get('description', [])]
                            # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                            for tag in range(len(description)):
                                ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                                if not isinstance(ttp, type(None)):
                                    if not os.path.exists(os.path.join(save_path_sigma_hq5, ttp)):
                                        os.makedirs(os.path.join(save_path_sigma_hq5, ttp))
                                else:
                                    ttp='T0000'
                                copy_file_to_path(item, save_path_sigma_hq5, ttp)
                else:
                    no_label_tags.append(item)
                    # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'tags' no exista.
                    if 'description' in document:
                        description = [document.get('description', [])]
                        # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                        for tag in range(len(description)):
                            ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                            if not isinstance(ttp, type(None)):
                                if not os.path.exists(os.path.join(save_path_sigma_hq5, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq5, ttp))
                            else:
                                ttp='T0000'
                            copy_file_to_path(item, save_path_sigma_hq5, ttp)
                    else:
                        copy_file_to_path(item, save_path_sigma_hq5, 'T0000')
            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except yaml.YAMLError as e:
                print(f"Error al cargar el archivo YAML: {e}")
                error_load.append(item)
                copy_file_to_path(item, save_path_sigma_hq5, 'T0000')
            
            # Con esta excepción llegamos si hemos recorrido la lista de tags y no hemos encontrado ninguna TTP, por lo que revisamos la etiqueta dando lugar a 3 casuísticas,
            # la primera que encuentre dicha etiqueta, en cuyo caso se revisa el contenido, si dentro de este se encuentra alguna TTP se mandara a la carpeta correspondiente. En caso de no haber coincidencia, al igual que ocurrirá si no hay etiqueta 'description' el fichero será copiado a T000.
            except NotTTPatTags as e:
                print(f"{e}")
                notTTPinTags.append(item)
                if 'description' in document:
                    description = [document.get('description', [])]
                    # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                    for tag in range(len(description)):
                        ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                        if not isinstance(ttp, type(None)):
                            notTTPinTags_with_TTP_in_description.append(item)
                            pass
                        else:
                            ttp='T0000'
                        copy_file_to_path(item, save_path_sigma_hq5, ttp)
                else:
                    notTTPinTags_without_description.append(item)
                    copy_file_to_path(item, save_path_sigma_hq5, 'T0000')

Error al cargar el archivo YAML: expected '<document start>', but found '<scalar>'
  in "C:\Users\jelopez\Downloads\socprime_sigma-master\Fancy_Bear\Rule\Fancy_Bear.yml", line 58, column 1
Copiado correctamente a la carpeta de la TTP: T0000
Error al cargar el archivo YAML: expected a single document in the stream
  in "C:\Users\jelopez\Downloads\socprime_sigma-master\FlawedAmmyy\Rule\FlawedAmmyy_RAT.yml", line 1, column 1
but found another document
  in "C:\Users\jelopez\Downloads\socprime_sigma-master\FlawedAmmyy\Rule\FlawedAmmyy_RAT.yml", line 21, column 1
Copiado correctamente a la carpeta de la TTP: T0000
.yaml sin TTP encontrada en los tags
Copiado correctamente a la carpeta de la TTP: T0000
Error al cargar el archivo YAML: while scanning for the next token
found character '\t' that cannot start any token
  in "C:\Users\jelopez\Downloads\socprime_sigma-master\InvisiMole\Rule\InvisiMole.yml", line 35, column 1
Copiado correctamente a la carpeta de la TTP: T0000
Error al cargar el a

In [69]:
ttps_in_tags_not_in_techniques_enterprise

['ATTACK.T1035', 'ATTACK.T1177']

In [63]:
print('Reglas totales para asignar: '+str(len(num_yamls)))
rules_unique = get_unique_items(num_yamls)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 12
Reglas únicas para asignar: 12


In [67]:
print('Reglas que retornan error de apertura: '+str(len(error_load)))

Reglas que retornan error de apertura: 10


In [64]:
print('Tienen informada la etiqueta "tags": '+str(len(has_label_tags)))
print('Tienen información en la etiqueta "tags": '+str(len(has_label_with_content_tags)))
print('NO tienen información en la etiqueta "tags": '+str(len(has_label_without_content_tags)))

Tienen informada la etiqueta "tags": 1
Tienen información en la etiqueta "tags": 1
NO tienen información en la etiqueta "tags": 0


In [65]:
print('NO tienen la etiqueta "tags": '+str(len(no_label_tags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags": '+str(len(notTTPinTags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": '+str(len(notTTPinTags_without_description)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": '+str(len(notTTPinTags_with_TTP_in_description)))

NO tienen la etiqueta "tags": 1
NO se ha encontrado TTP alguna en la etiqueta "tags": 1
NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": 0
NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": 0


In [66]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_sigma_hq5))))
print('Reglas sin TTP asignada (T0000): '+str(len(getListOfFilesSub(os.path.join(save_path_sigma_hq5, 'T0000')))))
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_sigma_hq5)) - len(getListOfFilesSub(os.path.join(save_path_sigma_hq5, 'T0000')))))
print('Reglas asignadas a 2 o más TTPs: '+str((len(getListOfFilesSub(save_path_sigma_hq5)) - len(getListOfFilesSub(os.path.join(save_path_sigma_hq5, 'T0000'))))-len(rules_unique)))

Número de TTPs únicas identificadas (puede incluir T0000): 1
Reglas sin TTP asignada (T0000): 12
Reglas asignadas a TTP: 0
Reglas asignadas a 2 o más TTPs: -12


#### TTP's Sigma HQ 6

**URL caída**

#### TTP's Sigma HQ 7

En este caso el repositorio almacena las reglas por técnica por lo que simplemente descargamos los ficheros y los movemos a la carpeta final

In [68]:
zip_sigma_hq7 = 'https://github.com/P4T12ICK/Sigma-Rule-Repository/archive/refs/heads/master.zip'
save_path_sigma_hq7 = os.path.join(save_path, 'sigmaHQ7')
unzip_folder = r'Sigma-Rule-Repository-master/detection-rules/'

In [69]:
download_folder_unzip(zip_sigma_hq7, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [70]:
copy_files_between_paths(os.path.join(temp_path, unzip_folder),save_path_sigma_hq7)

#### TTP's Sigma HQ 8

In [43]:
zip_sigma_hq8 = 'https://github.com/blacklanternsecurity/sigma-rules/archive/refs/heads/main.zip'
save_path_sigma_hq8 = os.path.join(save_path, 'sigmaHQ8')
unzip_folder = r'sigma-rules-main/'

In [51]:
download_unzip(zip_sigma_hq8, temp_path)

In [44]:
os.path.join(temp_path, unzip_folder)

'C:\\Users\\jelopez\\Downloads\\sigma-rules-main/'

In [45]:
files = getListOfFilesSub(os.path.join(temp_path, unzip_folder))
len(files)

11

In [47]:
create_output_folder(save_path_sigma_hq8)

In [57]:
num_yamls = []
has_label_tags = []
has_label_with_content_tags = []
has_label_without_content_tags = []
no_label_tags = []
notTTPinTags = []
notTTPinTags_with_TTP_in_description = []
notTTPinTags_without_description = []
error_load = []
error_load_ttp_in_path_name = []

ttps_in_tags_not_in_techniques_enterprise = []
ttps_in_tags_not_in_techniques_enterprise_items = []

for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yaml' or extension == '.yml':
        with open(item) as file:
            num_yamls.append(item)
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
                document = yaml.full_load(file)
                if 'tags' in document:
                    # Si encontramos la etiqueta 'tags' en el yaml lo guardamos como una variable 
                    tags = document.get('tags', [])
                    has_label_tags.append(item)
                    # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                    if not isinstance(tags, type(None)):
                        has_label_with_content_tags.append(item)
                        # Convertimos en lista el contenido de la etiqueta
                        #tags = [tags]
                        tags = [tag.upper() for tag in tags if isinstance(tag, str)]
                        # Recorremos la lista de items contenidos en tags y contamos las ttp encontradas ya que si no encontramos ninguna, pese haber texto en la etiqueta
                        # deberemos mover el fichero a la carpeta T0000
                        ttp_counter = 0
                        for tag in range(len(tags)):
                            # Matcheamos el item de la lista tags con nuestra lista de ttps 
                            ttp = find_techniques_in_list(tags[tag],techniques_enterprise)
                            # He encontrado en sigma HQ 5 tags que contienen técnicas que no están en el listado de techniques_enterprise, vamos a almacenarlas para reportarlas.
                            if '.T0' in tags[tag] or '.T1' in tags[tag]:
                                ttps_in_tags_not_in_techniques_enterprise.append(tags[tag])
                                ttps_in_tags_not_in_techniques_enterprise_items.append(item)
                            # Y si es un string lo que tenemos, entonces guardamos en la carpeta de la la ttp correspondiente
                            if isinstance(ttp, str):
                                ttp_counter += 1
                                copy_file_to_path(item, save_path_sigma_hq8, ttp)
                        if ttp_counter == 0:
                            raise NotTTPatTags('.yaml sin TTP encontrada en los tags')
                
                    else:
                        # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
                        has_label_without_content_tags.append(item)
                        if 'description' in document:
                            description = [document.get('description', [])]
                            # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                            for tag in range(len(description)):
                                ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                                if not isinstance(ttp, type(None)):
                                    if not os.path.exists(os.path.join(save_path_sigma_hq9, ttp)):
                                        os.makedirs(os.path.join(save_path_sigma_hq8, ttp))
                                else:
                                    ttp='T0000'
                                copy_file_to_path(item, save_path_sigma_hq8, ttp)
                else:
                    no_label_tags.append(item)
                    # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'tags' no exista.
                    if 'description' in document:
                        description = [document.get('description', [])]
                        # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                        for tag in range(len(description)):
                            ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                            if not isinstance(ttp, type(None)):
                                if not os.path.exists(os.path.join(save_path_sigma_hq8, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq8, ttp))
                            else:
                                ttp='T0000'
                            copy_file_to_path(item, save_path_sigma_hq8, ttp)
                    else:
                        copy_file_to_path(item, save_path_sigma_hq8, 'T0000')
            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except yaml.YAMLError as e:
                error_load.append(item)
                ttp = find_techniques_in_list(item,techniques_enterprise)
                if not isinstance(ttp, type(None)):
                    print(f'Error al cargar el archivo YAML: {e}\nEncontrada TTP en la path del fichero')
                    error_load_ttp_in_path_name.append(item)
                    copy_file_to_path(item, save_path_sigma_hq8, ttp)
                else:
                    print(f"Error al cargar el archivo YAML: {e}")
                    copy_file_to_path(item, save_path_sigma_hq8, 'T0000')
            # Con esta excepción llegamos si hemos recorrido la lista de tags y no hemos encontrado ninguna TTP, por lo que revisamos la etiqueta dando lugar a 3 casuísticas,
            # la primera que encuentre dicha etiqueta, en cuyo caso se revisa el contenido, si dentro de este se encuentra alguna TTP se mandara a la carpeta correspondiente. En caso de no haber coincidencia, al igual que ocurrirá si no hay etiqueta 'description' el fichero será copiado a T000.
            except NotTTPatTags as e:
                print(f"{e}")
                notTTPinTags.append(item)
                if 'description' in document:
                    description = [document.get('description', [])]
                    # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                    for tag in range(len(description)):
                        ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                        if not isinstance(ttp, type(None)):
                            notTTPinTags_with_TTP_in_description.append(item)
                            pass
                        else:
                            ttp='T0000'
                        copy_file_to_path(item, save_path_sigma_hq8, ttp)
                else:
                    notTTPinTags_without_description.append(item)
                    copy_file_to_path(item, save_path_sigma_hq8, 'T0000')

Copiado correctamente a la carpeta de la TTP: T1068
Copiado correctamente a la carpeta de la TTP: T1207
Copiado correctamente a la carpeta de la TTP: T1003.006
Copiado correctamente a la carpeta de la TTP: T1558.003
Copiado correctamente a la carpeta de la TTP: T1018
Copiado correctamente a la carpeta de la TTP: T1069.002
Copiado correctamente a la carpeta de la TTP: T1069.002
Copiado correctamente a la carpeta de la TTP: T1087.002
Copiado correctamente a la carpeta de la TTP: T1482
Error al cargar el archivo YAML: while scanning an alias
  in "C:\Users\jelopez\Downloads\sigma-rules-main/TA0008 - Lateral Movement\T1550 - Use Alternate Authentication Material\002 - Pass the Hash\ad_cs_relay.yml", line 18, column 25
expected alphabetic or numeric character, but found '\n'
  in "C:\Users\jelopez\Downloads\sigma-rules-main/TA0008 - Lateral Movement\T1550 - Use Alternate Authentication Material\002 - Pass the Hash\ad_cs_relay.yml", line 18, column 26
Encontrada TTP en la path del fichero
Co

In [54]:
print('Reglas totales para asignar: '+str(len(num_yamls)))
rules_unique = get_unique_items(num_yamls)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 10
Reglas únicas para asignar: 10


In [58]:
print('Reglas que retornan error de apertura: '+str(len(error_load)))
print('Reglas que en las que se ha identificado TTP en la path del archivo: '+str(len(error_load_ttp_in_path_name)))

Reglas que retornan error de apertura: 1
Reglas que en las que se ha identificado TTP en la path del archivo: 1


In [55]:
print('Tienen informada la etiqueta "tags": '+str(len(has_label_tags)))
print('Tienen información en la etiqueta "tags": '+str(len(has_label_with_content_tags)))
print('NO tienen información en la etiqueta "tags": '+str(len(has_label_without_content_tags)))

Tienen informada la etiqueta "tags": 9
Tienen información en la etiqueta "tags": 9
NO tienen información en la etiqueta "tags": 0


In [59]:
print('NO tienen la etiqueta "tags": '+str(len(no_label_tags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags": '+str(len(notTTPinTags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": '+str(len(notTTPinTags_without_description)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": '+str(len(notTTPinTags_with_TTP_in_description)))

NO tienen la etiqueta "tags": 0
NO se ha encontrado TTP alguna en la etiqueta "tags": 0
NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": 0
NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": 0


In [63]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_sigma_hq8))))
# print('Reglas sin TTP asignada (T0000): '+str(len(getListOfFilesSub(os.path.join(save_path_sigma_hq8, 'T0000')))))
try:
    t0000_count = len(os.listdir(getListOfFilesSub(os.path.join(save_path_sigma_hq8, 'T0000'))))
    print('Reglas sin TTP asignada (T0000):', t0000_count)
except FileNotFoundError:
    t0000_count = 0
    print('Reglas sin TTP asignada (T0000): 0')
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_sigma_hq8)) - t0000_count))
print('Reglas asignadas a 2 o más TTPs: '+str((len(getListOfFilesSub(save_path_sigma_hq8)) - t0000_count)-len(rules_unique)))

Número de TTPs únicas identificadas (puede incluir T0000): 9
Reglas sin TTP asignada (T0000): 0
Reglas asignadas a TTP: 10
Reglas asignadas a 2 o más TTPs: 0


#### TTP's Sigma HQ 9

<h5 style="background-color: #E11717; color: black;">
    Importante: No ejecutar por solicitud de Seguridad
</h5>

In [19]:
zip_sigma_hq9 = 'https://github.com/mdecrevoisier/SIGMA-detection-rules/archive/refs/heads/main.zip'
save_path_sigma_hq9 = os.path.join(save_path, 'sigmaHQ9')
unzip_folder = r'SIGMA-detection-rules-main/'

In [18]:
download_unzip(zip_sigma_hq9, temp_path)

In [ ]:
create_output_folder(save_path_sigma_hq9)

In [21]:
files = getListOfFilesSub(os.path.join(temp_path, unzip_folder))
len(files)

351

In [23]:
num_yamls = []
has_label_tags = []
has_label_with_content_tags = []
has_label_without_content_tags = []
no_label_tags = []
notTTPinTags = []
notTTPinTags_with_TTP_in_description = []
notTTPinTags_without_description = []
error_load = []
error_load_ttp_in_path_name = []

ttps_in_tags_not_in_techniques_enterprise = []
ttps_in_tags_not_in_techniques_enterprise_items = []

for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yaml' or extension == '.yml':
        with open(item) as file:
            num_yamls.append(item)
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
                document = yaml.full_load(file)
                if 'tags' in document:
                    # Si encontramos la etiqueta 'tags' en el yaml lo guardamos como una variable 
                    tags = document.get('tags', [])
                    has_label_tags.append(item)
                    # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                    if not isinstance(tags, type(None)):
                        has_label_with_content_tags.append(item)
                        # Convertimos en lista el contenido de la etiqueta
                        #tags = [tags]
                        tags = [tag.upper() for tag in tags if isinstance(tag, str)]
                        # Recorremos la lista de items contenidos en tags y contamos las ttp encontradas ya que si no encontramos ninguna, pese haber texto en la etiqueta
                        # deberemos mover el fichero a la carpeta T0000
                        ttp_counter = 0
                        for tag in range(len(tags)):
                            # Matcheamos el item de la lista tags con nuestra lista de ttps 
                            ttp = find_techniques_in_list(tags[tag],techniques_enterprise)
                            # He encontrado en sigma HQ 5 tags que contienen técnicas que no están en el listado de techniques_enterprise, vamos a almacenarlas para reportarlas.
                            if '.T0' in tags[tag] or '.T1' in tags[tag]:
                                ttps_in_tags_not_in_techniques_enterprise.append(tags[tag])
                                ttps_in_tags_not_in_techniques_enterprise_items.append(item)
                            # Y si es un string lo que tenemos, entonces guardamos en la carpeta de la la ttp correspondiente
                            if isinstance(ttp, str):
                                ttp_counter += 1
                                copy_file_to_path(item, save_path_sigma_hq9, ttp)
                        if ttp_counter == 0:
                            raise NotTTPatTags('.yaml sin TTP encontrada en los tags')
                
                    else:
                        # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
                        has_label_without_content_tags.append(item)
                        if 'description' in document:
                            description = [document.get('description', [])]
                            # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                            for tag in range(len(description)):
                                ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                                if not isinstance(ttp, type(None)):
                                    if not os.path.exists(os.path.join(save_path_sigma_hq9, ttp)):
                                        os.makedirs(os.path.join(save_path_sigma_hq9, ttp))
                                else:
                                    ttp='T0000'
                                copy_file_to_path(item, save_path_sigma_hq9, ttp)
                else:
                    no_label_tags.append(item)
                    # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'tags' no exista.
                    if 'description' in document:
                        description = [document.get('description', [])]
                        # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                        for tag in range(len(description)):
                            ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                            if not isinstance(ttp, type(None)):
                                if not os.path.exists(os.path.join(save_path_sigma_hq9, ttp)):
                                    os.makedirs(os.path.join(save_path_sigma_hq9, ttp))
                            else:
                                ttp='T0000'
                            copy_file_to_path(item, save_path_sigma_hq9, ttp)
                    else:
                        copy_file_to_path(item, save_path_sigma_hq9, 'T0000')
            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except yaml.YAMLError as e:
                error_load.append(item)
                ttp = find_techniques_in_list(item,techniques_enterprise)
                if not isinstance(ttp, type(None)):
                    print(f'Error al cargar el archivo YAML: {e}\nEncontrada TTP en la path del fichero')
                    error_load_ttp_in_path_name.append(item)
                    copy_file_to_path(item, save_path_sigma_hq9, ttp)
                else:
                    print(f"Error al cargar el archivo YAML: {e}")
                    copy_file_to_path(item, save_path_sigma_hq9, 'T0000')
            # Con esta excepción llegamos si hemos recorrido la lista de tags y no hemos encontrado ninguna TTP, por lo que revisamos la etiqueta dando lugar a 3 casuísticas,
            # la primera que encuentre dicha etiqueta, en cuyo caso se revisa el contenido, si dentro de este se encuentra alguna TTP se mandara a la carpeta correspondiente. En caso de no haber coincidencia, al igual que ocurrirá si no hay etiqueta 'description' el fichero será copiado a T000.
            except NotTTPatTags as e:
                print(f"{e}")
                notTTPinTags.append(item)
                if 'description' in document:
                    description = [document.get('description', [])]
                    # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                    for tag in range(len(description)):
                        ttp = find_techniques_in_list(description[tag],techniques_enterprise)
                        if not isinstance(ttp, type(None)):
                            notTTPinTags_with_TTP_in_description.append(item)
                            pass
                        else:
                            ttp='T0000'
                        copy_file_to_path(item, save_path_sigma_hq9, ttp)
                else:
                    notTTPinTags_without_description.append(item)
                    copy_file_to_path(item, save_path_sigma_hq9, 'T0000')

Copiado correctamente a la carpeta de la TTP: T1484.002
Copiado correctamente a la carpeta de la TTP: T1114.003
Copiado correctamente a la carpeta de la TTP: T1114
Copiado correctamente a la carpeta de la TTP: T1566
Copiado correctamente a la carpeta de la TTP: T1114
Copiado correctamente a la carpeta de la TTP: T1566
Copiado correctamente a la carpeta de la TTP: T1562.006
Copiado correctamente a la carpeta de la TTP: T1546
Copiado correctamente a la carpeta de la TTP: T1110.001
Copiado correctamente a la carpeta de la TTP: T1110.003
Copiado correctamente a la carpeta de la TTP: T1136
Copiado correctamente a la carpeta de la TTP: T1098
Copiado correctamente a la carpeta de la TTP: T1136
Copiado correctamente a la carpeta de la TTP: T1068
Copiado correctamente a la carpeta de la TTP: T1222.001
Copiado correctamente a la carpeta de la TTP: T1098
Copiado correctamente a la carpeta de la TTP: T1036
Copiado correctamente a la carpeta de la TTP: T1068
Copiado correctamente a la carpeta de la

In [27]:
print('Reglas totales para asignar: '+str(len(num_yamls)))
rules_unique = get_unique_items(num_yamls)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 348
Reglas únicas para asignar: 348


In [28]:
print('Reglas que retornan error de apertura: '+str(len(error_load)))
print('Reglas que en las que se ha identificado TTP en la path del archivo: '+str(len(error_load_ttp_in_path_name)))

Reglas que retornan error de apertura: 1
Reglas que en las que se ha identificado TTP en la path del archivo: 0


In [29]:
print('Tienen informada la etiqueta "tags": '+str(len(has_label_tags)))
print('Tienen información en la etiqueta "tags": '+str(len(has_label_with_content_tags)))
print('NO tienen información en la etiqueta "tags": '+str(len(has_label_without_content_tags)))

Tienen informada la etiqueta "tags": 347
Tienen información en la etiqueta "tags": 347
NO tienen información en la etiqueta "tags": 0


In [30]:
print('NO tienen la etiqueta "tags": '+str(len(no_label_tags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags": '+str(len(notTTPinTags)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": '+str(len(notTTPinTags_without_description)))
print('NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": '+str(len(notTTPinTags_with_TTP_in_description)))

NO tienen la etiqueta "tags": 0
NO se ha encontrado TTP alguna en la etiqueta "tags": 4
NO se ha encontrado TTP alguna en la etiqueta "tags" y NO disponen de la etiqueta "description": 0
NO se ha encontrado TTP alguna en la etiqueta "tags" y disponen de la etiqueta "description": 0


In [38]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_sigma_hq9))))
# print('Reglas sin TTP asignada (T0000): '+str(len(getListOfFilesSub(os.path.join(save_path_sigma_hq8, 'T0000')))))
try:
    t0000_count = len(getListOfFilesSub(os.path.join(save_path_sigma_hq9, 'T0000')))
    print('Reglas sin TTP asignada (T0000):', t0000_count)
except FileNotFoundError:
    t0000_count = 0
    print('Reglas sin TTP asignada (T0000): 0')
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_sigma_hq9)) - t0000_count))
print('Reglas asignadas a 2 o más TTPs: '+str((len(getListOfFilesSub(save_path_sigma_hq9)) - t0000_count)-len(rules_unique)))


Número de TTPs únicas identificadas (puede incluir T0000): 111
Reglas sin TTP asignada (T0000): 5
Reglas asignadas a TTP: 410
Reglas asignadas a 2 o más TTPs: 62


#### TTP's Yara 1

In [18]:
zip_yara_1 = 'https://github.com/Yara-Rules/rules/archive/refs/heads/master.zip'
save_path_yara_1 = os.path.join(save_path, 'yara_1')
unzip_folder = r'rules-master/'

In [19]:
download_unzip(zip_yara_1, temp_path)

In [20]:
create_output_folder(save_path_yara_1)

In [21]:
files = getListOfFilesSub(os.path.join(temp_path, unzip_folder))
len(files)

582

In [22]:
num_yaras = []
ttp_in_filename = []
content_in_yara = []
no_content_in_yara = []
ttp_in_yara = []
error_load = []
no_ttp_in_content_or_filename = []


for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yar' or extension == '.yara':
        with open(item, 'r', encoding='utf-8') as file:
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                md_content = file.read()
                md_content = md_content.upper()
                num_yaras.append(item)
                # Revisamos que se haya podido obtener texto del contenido del .yar 
                if isinstance(md_content, str):
                    ttps = find_techniques(md_content)
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    content_in_yara.append(item)
                else:
                    #Si el contenido no dispone de texto, buscamos ttps en el propio nombre del fichero
                    ttps = find_techniques(item.upper())
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    no_content_in_yara.append(item)
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_yara.append(item)
                            copy_file_to_path(item, save_path_yara_1, ttp)
                
                #En cualquier caso, siempre buscamos ttp en el nombre del fichero
                ttps = find_techniques(item.upper())
                ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_filename.append(item)
                            copy_file_to_path(item, save_path_yara_1, ttp)
                else:
                    copy_file_to_path(item, save_path_yara_1, 'T0000')

            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except Exception as e:
                print(f"Error al intentar leer el archivo markdown: {e}")
                error_load.append(item)
                ttps = find_techniques(item.upper())
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_filename.append(item)
                            copy_file_to_path(item, save_path_yara_1, ttp)
                else:
                    copy_file_to_path(item, save_path_yara_1, 'T0000')

Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado corr

In [23]:
print('Reglas totales para asignar: '+str(len(num_yaras)))
rules_unique = get_unique_items(num_yaras)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 566
Reglas únicas para asignar: 566


In [24]:
print('Reglas que retornan error de lectura: '+str(len(error_load)))

Reglas que retornan error de lectura: 0


In [25]:
print('El archivo ha sido abierto y dispone de contenido: '+str(len(content_in_yara)))
print('El archivo ha sido abierto y NO dispone de contenido: '+str(len(no_content_in_yara)))
print('Se ha podido identificar al menos una TTP en el contenido del archivo: '+str(len(ttp_in_yara)))
print('Se ha podido identificar al menos una TTP en el nombre del archivo: '+str(len(ttp_in_filename)))

El archivo ha sido abierto y dispone de contenido: 566
El archivo ha sido abierto y NO dispone de contenido: 0
Se ha podido identificar al menos una TTP en el contenido del archivo: 0
Se ha podido identificar al menos una TTP en el nombre del archivo: 0


In [26]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_yara_1))))
try:
    t0000_count = len(getListOfFilesSub(os.path.join(save_path_yara_1, 'T0000')))
    print('Reglas sin TTP asignada (T0000):', t0000_count)
except FileNotFoundError:
    t0000_count = 0
    print('Reglas sin TTP asignada (T0000): 0')
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_yara_1)) - t0000_count))
print('Reglas asignadas a 2 o más TTP: '+str(len(count_duplicate_rules(getListOfFilesSub(save_path_yara_1)))))

Número de TTPs únicas identificadas (puede incluir T0000): 1
Reglas sin TTP asignada (T0000): 566
Reglas asignadas a TTP: 0
Reglas asignadas a 2 o más TTP: 0


#### TTP's Yara 2

In [19]:
zip_yara_2 = 'https://github.com/bartblaze/Yara-rules/archive/refs/heads/master.zip'
save_path_yara_2 = os.path.join(save_path, 'yara_2')
unzip_folder = r'Yara-rules-master/'

In [20]:
download_unzip(zip_yara_2, temp_path)

In [21]:
create_output_folder(save_path_yara_2)

In [22]:
files = getListOfFilesSub(os.path.join(temp_path, unzip_folder))
len(files)

93

In [23]:
num_yaras = []
ttp_in_filename = []
content_in_yara = []
no_content_in_yara = []
ttp_in_yara = []
error_load = []
no_ttp_in_content_or_filename = []


for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.yar' or extension == '.yara':
        with open(item, 'r', encoding='utf-8') as file:
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                md_content = file.read()
                md_content = md_content.upper()
                num_yaras.append(item)
                # Revisamos que se haya podido obtener texto del contenido del .yar 
                if isinstance(md_content, str):
                    ttps = find_techniques(md_content)
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    content_in_yara.append(item)
                else:
                    #Si el contenido no dispone de texto, buscamos ttps en el propio nombre del fichero
                    ttps = find_techniques(item.upper())
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    no_content_in_yara.append(item)
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_yara.append(item)
                            copy_file_to_path(item, save_path_yara_2, ttp)
                
                #En cualquier caso, siempre buscamos ttp en el nombre del fichero
                ttps = find_techniques(item.upper())
                ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_filename.append(item)
                            copy_file_to_path(item, save_path_yara_2, ttp)
                else:
                    copy_file_to_path(item, save_path_yara_2, 'T0000')

            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except Exception as e:
                print(f"Error al intentar leer el archivo markdown: {e}")
                error_load.append(item)
                ttps = find_techniques(item.upper())
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_filename.append(item)
                            copy_file_to_path(item, save_path_yara_2, ttp)
                else:
                    copy_file_to_path(item, save_path_yara_2, 'T0000')

Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado corr

In [24]:
print('Reglas totales para asignar: '+str(len(num_yaras)))
rules_unique = get_unique_items(num_yaras)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 88
Reglas únicas para asignar: 88


In [25]:
print('Reglas que retornan error de lectura: '+str(len(error_load)))

Reglas que retornan error de lectura: 0


In [26]:
print('El archivo ha sido abierto y dispone de contenido: '+str(len(content_in_yara)))
print('El archivo ha sido abierto y NO dispone de contenido: '+str(len(no_content_in_yara)))
print('Se ha podido identificar al menos una TTP en el contenido del archivo: '+str(len(ttp_in_yara)))
print('Se ha podido identificar al menos una TTP en el nombre del archivo: '+str(len(ttp_in_filename)))

El archivo ha sido abierto y dispone de contenido: 88
El archivo ha sido abierto y NO dispone de contenido: 0
Se ha podido identificar al menos una TTP en el contenido del archivo: 2
Se ha podido identificar al menos una TTP en el nombre del archivo: 0


In [29]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_yara_2))))
try:
    t0000_count = len(getListOfFilesSub(os.path.join(save_path_yara_2, 'T0000')))
    print('Reglas sin TTP asignada (T0000):', t0000_count)
except FileNotFoundError:
    t0000_count = 0
    print('Reglas sin TTP asignada (T0000): 0')
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_yara_2)) - t0000_count))
print('Reglas asignadas a 2 o más TTP: '+str(len(count_duplicate_rules(getListOfFilesSub(save_path_yara_2)))))

Número de TTPs únicas identificadas (puede incluir T0000): 3
Reglas sin TTP asignada (T0000): 88
Reglas asignadas a TTP: 2
Reglas asignadas a 2 o más TTP: 1


#### TTP's Atomic threat 

<h5 style="background-color: #F80000; color: black;">
    Importante: No ejecutar por solicitud de Seguridad
</h5>


In [37]:
# zip_atomic = 'https://github.com/krakow2600/atomic-threat-coverage/archive/refs/heads/master.zip'
save_path_atomic = os.path.join(save_path, 'atomic')
unzip_folder = r'atomic-threat-coverage-master/Atomic_Threat_Coverage/'

In [38]:
download_folder_unzip(zip_atomic, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [39]:
create_output_folder(save_path_atomic)

In [65]:
files = getListOfFilesSub(os.path.join(temp_path, unzip_folder))
len(files)

534

In [75]:
num_mds = []
ttp_in_filename = []
content_in_md = []
no_content_in_md = []
ttp_in_md = []
error_load = []
no_ttp_in_content_or_filename = []

for item in files:
    root, extension = os.path.splitext(item)
    if extension == '.md' or extension == '.markdown':
        with open(item, 'r', encoding='utf-8') as file:
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                md_content = file.read()
                md_content = md_content.upper()
                num_mds.append(item)
                # Revisamos que se haya podido obtener texto del contenido del .md 
                if isinstance(md_content, str):
                    ttps = find_techniques(md_content)
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    content_in_md.append(item)
                else:
                    #Si el contenido no dispone de texto, buscamos ttps en el propio nombre del fichero
                    ttps = find_techniques(item.upper())
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    no_content_in_md.append(item)
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_md.append(item)
                            copy_file_to_path(item, save_path_atomic, ttp)
                
                #En cualquier caso, siempre buscamos ttp en el nombre del fichero
                ttps = find_techniques(item.upper())
                ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_filename.append(item)
                            copy_file_to_path(item, save_path_atomic, ttp)
                else:
                    copy_file_to_path(item, save_path_atomic, 'T0000')

            # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
            except Exception as e:
                print(f"Error al intentar leer el archivo markdown: {e}")
                error_load.append(item)
                ttps = find_techniques(item.upper())
                if len(ttps)>0:
                    # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                    for ttp in ttps:
                        if ttp != '':
                            ttp_in_md.append(item)
                            copy_file_to_path(item, save_path_atomic, ttp)
                else:
                    copy_file_to_path(item, save_path_atomic, 'T0000')

Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado correctamente a la carpeta de la TTP: T0000
Copiado corr

In [76]:
print('Reglas totales para asignar: '+str(len(num_mds)))
rules_unique = get_unique_items(num_mds)
print('Reglas únicas para asignar: '+str(len(rules_unique)))

Reglas totales para asignar: 534
Reglas únicas para asignar: 534


In [77]:
print('Reglas que retornan error de lectura: '+str(len(error_load)))

Reglas que retornan error de lectura: 0


In [86]:
print('El archivo ha sido abierto y dispone de contenido: '+str(len(content_in_md)))
print('El archivo ha sido abierto y NO dispone de contenido: '+str(len(no_content_in_md)))
print('Se ha podido identificar al menos una TTP en el contenido del archivo: '+str(len(ttp_in_md)))
print('Se ha podido identificar al menos una TTP en el nombre del archivo: '+str(len(ttp_in_filename)))

El archivo ha sido abierto y dispone de contenido: 534
El archivo ha sido abierto y NO dispone de contenido: 0
Se ha podido identificar al menos una TTP en el contenido del archivo: 247
Se ha podido identificar al menos una TTP en el nombre del archivo: 63


In [114]:
print('Número de TTPs únicas identificadas (puede incluir T0000): '+str(len(get_folders(save_path_atomic))))
# print('Reglas sin TTP asignada (T0000): '+str(len(getListOfFilesSub(os.path.join(save_path_sigma_hq8, 'T0000')))))
try:
    t0000_count = len(getListOfFilesSub(os.path.join(save_path_atomic, 'T0000')))
    print('Reglas sin TTP asignada (T0000):', t0000_count)
except FileNotFoundError:
    t0000_count = 0
    print('Reglas sin TTP asignada (T0000): 0')
print('Reglas asignadas a TTP: '+str(len(getListOfFilesSub(save_path_atomic)) - t0000_count))
print('Reglas asignadas a 2 o más TTP: '+str(len(count_duplicate_rules(getListOfFilesSub(save_path_atomic)))))

Número de TTPs únicas identificadas (puede incluir T0000): 78
Reglas sin TTP asignada (T0000): 471
Reglas asignadas a TTP: 247
Reglas asignadas a 2 o más TTP: 150
